# 01. Setup & Environment (라이브러리 및 GPU 설정)

In [1]:
"""
============================================================
  project_setup.py
  ML/DL 프로젝트 시작 시 1회 실행하는 최적화 설정 파일
  대상 환경: Windows 11 Pro
             Intel i9-14900K | RTX 4070 12GB
             CUDA 12.4 | cuDNN 9.19.0 | Python 3.12.10
============================================================
사용법:
    import project_setup          # 자동 실행
    또는
    from project_setup import setup
    cfg = setup()
============================================================
"""

import os
import sys
import platform
import warnings
import logging
import time

# ─────────────────────────────────────────────────────────
# 내부 출력 헬퍼
# ─────────────────────────────────────────────────────────
def _log(msg: str, level: str = "ok"):
    icons = {"ok": "✅", "warn": "⚠️ ", "err": "❌", "info": "📌", "title": "🚀"}
    print(f"  {icons.get(level, ' ')} {msg}")


# ─────────────────────────────────────────────────────────
# 1. 경고 & 로깅 설정
# ─────────────────────────────────────────────────────────
def _configure_warnings():
    # 불필요한 경고 억제
    warnings.filterwarnings("ignore", category=UserWarning)
    warnings.filterwarnings("ignore", category=FutureWarning)
    warnings.filterwarnings("ignore", category=DeprecationWarning)

    # 외부 라이브러리 경고 억제
    os.environ["TF_CPP_MIN_LOG_LEVEL"]        = "3"
    os.environ["TRANSFORMERS_VERBOSITY"]       = "error"
    os.environ["TOKENIZERS_PARALLELISM"]       = "false"
    os.environ["DATASETS_VERBOSITY"]           = "error"

    # Python 기본 로거 레벨
    logging.basicConfig(
        level=logging.WARNING,
        format="%(asctime)s [%(levelname)s] %(message)s",
        datefmt="%H:%M:%S",
    )
    # 외부 라이브러리 로거 억제
    for noisy in ["transformers", "datasets", "PIL", "matplotlib",
                  "urllib3", "filelock", "huggingface_hub"]:
        logging.getLogger(noisy).setLevel(logging.ERROR)

    _log("경고 & 로깅 설정 완료")


# ─────────────────────────────────────────────────────────
# 2. Windows 11 멀티프로세싱 설정
# ─────────────────────────────────────────────────────────
def _configure_multiprocessing():
    """
    Windows에서 DataLoader num_workers > 0 사용 시
    반드시 if __name__ == '__main__' 블록이 필요하거나,
    spawn context를 명시해야 함.
    여기서는 환경 변수만 최적화.
    """
    # i9-14900K: P코어 8개(HT=16) + E코어 16개 = 32스레드
    # DataLoader에 적합한 worker 수 (보통 P코어 수 = 8)
    cpu_count = os.cpu_count() or 8
    optimal_workers = min(8, cpu_count // 2)

    os.environ["OMP_NUM_THREADS"]      = str(optimal_workers)
    os.environ["MKL_NUM_THREADS"]      = str(optimal_workers)
    os.environ["OPENBLAS_NUM_THREADS"] = str(optimal_workers)
    os.environ["NUMEXPR_NUM_THREADS"]  = str(optimal_workers)

    _log(f"CPU 멀티스레드 설정 완료 (threads={optimal_workers}, cpu_total={cpu_count})")
    return optimal_workers


# ─────────────────────────────────────────────────────────
# 3. CUDA 활성화 & 컨텍스트 초기화
# ─────────────────────────────────────────────────────────
def activate_cuda(verbose: bool = True) -> dict:
    """
    CUDA 컨텍스트를 명시적으로 초기화하고 활성 상태를 검증합니다.

    일반적으로 첫 번째 CUDA 연산 시 자동 초기화되지만,
    이 함수를 호출하면:
      - 드라이버 레벨 초기화를 선행 완료
      - 첫 번째 실제 연산의 지연(latency) 제거
      - CUDA / cuDNN / 연산 파이프라인 이상 여부 사전 점검

    Returns
    -------
    dict — CUDA 활성화 결과 및 상세 정보
    """
    result = {
        "cuda_available"    : False,
        "context_initialized": False,
        "warmup_passed"     : False,
        "cuda_version"      : None,
        "cudnn_version"     : None,
        "driver_version"    : None,
        "device_name"       : None,
        "compute_capability": None,
        "vram_total_gb"     : None,
        "vram_free_gb"      : None,
    }

    if verbose:
        print()
        print("  ┌─────────────── CUDA 활성화 & 검증 ────────────────┐")

    try:
        import torch

        # ── Step 1: 기본 가용성 체크 ──────────────────────────
        if not torch.cuda.is_available():
            _log("CUDA 사용 불가 — 드라이버/설치 상태를 확인하세요", "err")
            if verbose:
                print("  └────────────────────────────────────────────────────┘")
            return result

        result["cuda_available"] = True
        result["cuda_version"]   = torch.version.cuda
        result["cudnn_version"]  = torch.backends.cudnn.version()

        # ── Step 2: 드라이버 버전 확인 ────────────────────────
        try:
            import subprocess
            smi = subprocess.run(
                ["nvidia-smi", "--query-gpu=driver_version",
                 "--format=csv,noheader"],
                capture_output=True, text=True, timeout=5
            )
            if smi.returncode == 0:
                result["driver_version"] = smi.stdout.strip()
        except Exception:
            result["driver_version"] = "확인 불가"

        # ── Step 3: 컨텍스트 초기화 (핵심) ───────────────────
        #   torch.cuda.init() : CUDA 런타임 컨텍스트 명시적 생성
        #   이후 연산에서 발생하는 ~1~2초 지연을 여기서 선 처리
        torch.cuda.init()
        torch.cuda.set_device(0)               # GPU 0번 명시적 선택
        result["context_initialized"] = True

        # ── Step 4: GPU 상세 정보 수집 ────────────────────────
        props = torch.cuda.get_device_properties(0)
        free_mem, total_mem = torch.cuda.mem_get_info(0)

        result.update({
            "device_name"       : torch.cuda.get_device_name(0),
            "compute_capability": f"{props.major}.{props.minor}",
            "vram_total_gb"     : round(total_mem / 1024**3, 2),
            "vram_free_gb"      : round(free_mem  / 1024**3, 2),
            "sm_count"          : props.multi_processor_count,
        })

        # ── Step 5: Warm-up 커널 실행 ─────────────────────────
        #   실제 CUDA 커널을 실행해 파이프라인 이상 여부 확인
        #   (1) 소형 텐서 할당 / 연산 / 동기화
        #   (2) cuDNN 컨볼루션 커널 실행 (cuDNN 컨텍스트 초기화)
        #   (3) Mixed Precision 연산 확인
        device = torch.device("cuda:0")

        # Warm-up 1: 기본 CUDA 연산
        _a = torch.ones(512, 512, device=device)
        _b = torch.ones(512, 512, device=device)
        _c = torch.mm(_a, _b)
        torch.cuda.synchronize()

        # Warm-up 2: cuDNN 컨볼루션 커널 초기화
        _conv_in  = torch.randn(1, 3, 64, 64, device=device)
        _conv     = torch.nn.Conv2d(3, 16, 3, padding=1).to(device)
        _conv_out = _conv(_conv_in)
        torch.cuda.synchronize()

        # Warm-up 3: Mixed Precision (FP16) 경로 확인
        _a16 = _a.half()
        _b16 = _b.half()
        _c16 = torch.mm(_a16, _b16)
        torch.cuda.synchronize()

        # Warm-up 텐서 정리
        del _a, _b, _c, _a16, _b16, _c16
        del _conv_in, _conv, _conv_out
        torch.cuda.empty_cache()

        result["warmup_passed"] = True

        # ── Step 6: 결과 출력 ─────────────────────────────────
        if verbose:
            ok  = "✅"
            vram_used = result["vram_total_gb"] - result["vram_free_gb"]
            bar_len   = 25
            ratio     = vram_used / result["vram_total_gb"]
            filled    = int(bar_len * ratio)
            bar       = "█" * filled + "░" * (bar_len - filled)

            print(f"  │  {ok} CUDA {result['cuda_version']}  |  "
                  f"cuDNN {result['cudnn_version']}  |  "
                  f"Driver {result['driver_version']}")
            print(f"  │  {ok} GPU   : {result['device_name']}")
            print(f"  │  {ok} CC    : {result['compute_capability']}  |  "
                  f"SM: {result['sm_count']}개")
            print(f"  │  {ok} VRAM  : [{bar}] "
                  f"{vram_used:.2f}/{result['vram_total_gb']:.2f} GB "
                  f"(여유 {result['vram_free_gb']:.2f} GB)")
            print(f"  │  {ok} Context 초기화 완료")
            print(f"  │  {ok} Warm-up 완료 (CUDA / cuDNN / FP16)")
            print("  └────────────────────────────────────────────────────┘")
            print()

    except RuntimeError as e:
        _log(f"CUDA 활성화 실패: {e}", "err")
        if verbose:
            print("  └────────────────────────────────────────────────────┘")

    except ImportError:
        _log("PyTorch 미설치 → CUDA 활성화 불가", "err")
        if verbose:
            print("  └────────────────────────────────────────────────────┘")

    return result


# ─────────────────────────────────────────────────────────
# 3-B. CUDA / PyTorch 최적화
# ─────────────────────────────────────────────────────────
def _configure_pytorch():
    info = {}
    try:
        import torch

        # ── GPU 선택 ─────────────────────────────────────
        os.environ["CUDA_DEVICE_ORDER"]   = "PCI_BUS_ID"
        os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # RTX 4070 단일 GPU

        # ── VRAM 단편화 방지 (12GB에 최적화) ─────────────
        os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
            "max_split_size_mb:512,"
            "expandable_segments:True,"
            "garbage_collection_threshold:0.8"
        )

        if not torch.cuda.is_available():
            _log("CUDA 사용 불가 → CPU 모드", "warn")
            info["device"] = torch.device("cpu")
            return info

        device = torch.device("cuda:0")

        # ── cuDNN 최적화 ──────────────────────────────────
        torch.backends.cudnn.enabled        = True
        torch.backends.cudnn.benchmark      = True   # 입력 크기 고정 시 최고 속도
        torch.backends.cudnn.deterministic  = False  # benchmark=True와 병행 시 False 권장
        torch.backends.cudnn.allow_tf32     = True   # Ampere 이상 TF32 가속

        # ── TF32 (RTX 4070 Ada 아키텍처 가속) ─────────────
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = True

        # ── 메모리 사전 준비 (단편화 방지용 warm-up) ──────
        _dummy = torch.zeros(1, device=device)
        del _dummy
        torch.cuda.empty_cache()

        # ── 정보 수집 ─────────────────────────────────────
        props = torch.cuda.get_device_properties(0)
        total_vram = props.total_memory / 1024**3

        info = {
            "device"       : device,
            "gpu_name"     : torch.cuda.get_device_name(0),
            "cuda_version" : torch.version.cuda,
            "cudnn_version": torch.backends.cudnn.version(),
            "vram_total_gb": round(total_vram, 2),
            "torch_version": torch.__version__,
            "compute_cap"  : f"{props.major}.{props.minor}",
            "sm_count"     : props.multi_processor_count,
        }

        _log(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda} | "
             f"cuDNN {torch.backends.cudnn.version()}")
        _log(f"GPU: {info['gpu_name']} | VRAM: {total_vram:.2f} GB | "
             f"SM: {props.multi_processor_count} | CC: {props.major}.{props.minor}")
        _log("cuDNN benchmark=True | TF32=True | VRAM 단편화 방지 설정 완료")

    except ImportError:
        _log("PyTorch 미설치", "err")

    return info


# ─────────────────────────────────────────────────────────
# 4. NumPy / Pandas 최적화
# ─────────────────────────────────────────────────────────
def _configure_numpy_pandas():
    try:
        import numpy as np
        # MKL/OpenBLAS 사용 확인
        np_config = np.__config__
        _log(f"NumPy {np.__version__} 로드 완료")
    except ImportError:
        _log("NumPy 미설치", "err")

    try:
        import pandas as pd
        # Copy-on-Write (pandas 2.0+) 활성화 → 메모리 효율 향상
        pd.options.mode.copy_on_write            = True
        pd.options.display.max_columns           = 50
        pd.options.display.max_rows              = 100
        pd.options.display.float_format          = "{:.4f}".format
        pd.options.display.max_colwidth          = 80
        _log(f"Pandas {pd.__version__} | Copy-on-Write=True 설정 완료")
    except ImportError:
        _log("Pandas 미설치", "err")


# ─────────────────────────────────────────────────────────
# 5. Matplotlib 설정
# ─────────────────────────────────────────────────────────
def _configure_matplotlib():
    try:
        import matplotlib
        matplotlib.use("Agg")                          # Windows headless 안전 백엔드
        import matplotlib.pyplot as plt
        plt.rcParams.update({
            "figure.figsize"     : (12, 6),
            "figure.dpi"         : 120,
            "axes.grid"          : True,
            "grid.alpha"         : 0.3,
            "axes.spines.top"    : False,
            "axes.spines.right"  : False,
            "font.size"          : 11,
        })
        _log(f"Matplotlib {matplotlib.__version__} | Agg 백엔드 설정 완료")
    except ImportError:
        _log("Matplotlib 미설치", "err")


# ─────────────────────────────────────────────────────────
# 6. 재현성(Seed) 고정 유틸
# ─────────────────────────────────────────────────────────
def set_seed(seed: int = 42):
    """
    모든 난수 시드를 고정합니다.
    완전한 재현성이 필요할 때 호출하세요.

    주의: cuDNN deterministic=True로 전환되어 속도가 약간 감소할 수 있습니다.
    """
    import random
    random.seed(seed)

    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass

    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark     = False   # deterministic 시 False
    except ImportError:
        pass

    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"  🎲 Seed {seed} 고정 완료 (random / numpy / torch / PYTHONHASHSEED)")


# ─────────────────────────────────────────────────────────
# 7. GPU 메모리 상태 출력 유틸
# ─────────────────────────────────────────────────────────
def gpu_memory_status():
    """현재 GPU 메모리 사용량을 출력합니다."""
    try:
        import torch
        if not torch.cuda.is_available():
            print("  ⚠️  CUDA 사용 불가")
            return
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved  = torch.cuda.memory_reserved()  / 1024**3
        total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
        free      = total - allocated
        bar_len   = 30
        used_ratio = allocated / total
        filled    = int(bar_len * used_ratio)
        bar       = "█" * filled + "░" * (bar_len - filled)
        print(f"\n  💾 GPU 메모리 현황 [{bar}] {used_ratio*100:.1f}%")
        print(f"     사용중: {allocated:.2f} GB  |  예약: {reserved:.2f} GB  "
              f"|  여유: {free:.2f} GB  |  전체: {total:.2f} GB")
    except ImportError:
        print("  ❌ PyTorch 미설치")


# ─────────────────────────────────────────────────────────
# 8. GPU 메모리 정리 유틸
# ─────────────────────────────────────────────────────────
def clear_gpu_memory():
    """GPU 캐시를 비우고 가비지 컬렉션을 실행합니다."""
    import gc
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
            print("  🧹 GPU 메모리 캐시 정리 완료")
    except ImportError:
        pass


# ─────────────────────────────────────────────────────────
# 9. 권장 하이퍼파라미터 프리셋 (RTX 4070 기준)
# ─────────────────────────────────────────────────────────
def get_recommended_config(task: str = "image_classification") -> dict:
    """
    RTX 4070 12GB 기준 작업별 권장 설정을 반환합니다.

    task 옵션:
        "image_classification" | "object_detection" |
        "segmentation"         | "nlp_bert"         |
        "nlp_llm"              | "tabular"
    """
    base = {
        "device"       : "cuda",
        "num_workers"  : 8,          # i9-14900K P코어 기준
        "pin_memory"   : True,       # CPU→GPU 전송 속도 향상
        "prefetch_factor": 2,
        "persistent_workers": True,
        "use_amp"      : True,       # Mixed Precision (FP16)
    }

    presets = {
        "image_classification": {
            **base,
            "batch_size"   : 64,
            "image_size"   : 224,
            "optimizer"    : "AdamW",
            "lr"           : 1e-4,
            "weight_decay" : 1e-2,
            "epochs"       : 50,
            "scheduler"    : "CosineAnnealingLR",
            "note"         : "ResNet50/EfficientNet/ViT 기준",
        },
        "object_detection": {
            **base,
            "batch_size"   : 16,
            "image_size"   : 640,
            "optimizer"    : "SGD",
            "lr"           : 1e-2,
            "epochs"       : 100,
            "note"         : "YOLOv8/Faster-RCNN 기준",
        },
        "segmentation": {
            **base,
            "batch_size"   : 8,
            "image_size"   : 512,
            "optimizer"    : "AdamW",
            "lr"           : 3e-4,
            "epochs"       : 100,
            "note"         : "U-Net/SegFormer 기준",
        },
        "nlp_bert": {
            **base,
            "batch_size"   : 32,
            "max_length"   : 512,
            "optimizer"    : "AdamW",
            "lr"           : 2e-5,
            "weight_decay" : 1e-2,
            "epochs"       : 5,
            "warmup_ratio" : 0.1,
            "note"         : "BERT/RoBERTa 파인튜닝 기준",
        },
        "nlp_llm": {
            **base,
            "batch_size"   : 4,
            "gradient_accumulation_steps": 8,   # 유효 배치 = 32
            "max_length"   : 2048,
            "optimizer"    : "AdamW",
            "lr"           : 2e-4,
            "epochs"       : 3,
            "use_lora"     : True,
            "lora_r"       : 16,
            "lora_alpha"   : 32,
            "note"         : "LLM LoRA 파인튜닝 기준 (12GB 한계 내)",
        },
        "tabular": {
            "device"       : "cuda",
            "batch_size"   : 4096,
            "optimizer"    : "AdamW",
            "lr"           : 1e-3,
            "epochs"       : 100,
            "num_workers"  : 4,
            "note"         : "TabNet/MLP 기준",
        },
    }

    cfg = presets.get(task, base)
    print(f"\n  📋 [{task}] 권장 설정")
    for k, v in cfg.items():
        print(f"     {k:<35} = {v}")
    return cfg


# ─────────────────────────────────────────────────────────
# 10. Mixed Precision Scaler 팩토리
# ─────────────────────────────────────────────────────────
def get_amp_scaler():
    """
    RTX 4070용 Mixed Precision GradScaler를 반환합니다.

    사용 예:
        scaler = get_amp_scaler()
        with torch.autocast(device_type='cuda'):
            loss = model(x)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    """
    try:
        import torch
        if torch.cuda.is_available():
            scaler = torch.amp.GradScaler(
                init_scale=2.**16,
                growth_factor=2.0,
                backoff_factor=0.5,
                growth_interval=2000,
                enabled=True,
            )
            _log("GradScaler (Mixed Precision FP16) 준비 완료")
            return scaler
    except ImportError:
        _log("PyTorch 미설치 → GradScaler 반환 불가", "err")
    return None


# ─────────────────────────────────────────────────────────
# 메인 setup() 함수
# ─────────────────────────────────────────────────────────
def setup(seed: int | None = None, verbose: bool = True) -> dict:
    """
    프로젝트 시작 시 모든 최적화 설정을 한 번에 실행합니다.

    Parameters
    ----------
    seed    : int | None — 지정 시 재현성 고정 (기본: None = 고정 안 함)
    verbose : bool       — 시스템 요약 출력 여부

    Returns
    -------
    dict — device, gpu_name, vram 등 주요 정보
    """
    t_start = time.time()

    print()
    print("  ╔══════════════════════════════════════════════════╗")
    print("  ║      ML / DL 프로젝트 환경 초기화 중...         ║")
    print("  ║  i9-14900K | RTX 4070 12GB | CUDA 12.4          ║")
    print("  ╚══════════════════════════════════════════════════╝")
    print()

    _configure_warnings()
    workers   = _configure_multiprocessing()
    cuda_info = activate_cuda(verbose=True)     # CUDA 활성화 & 검증
    gpu_info  = _configure_pytorch()            # PyTorch 세부 최적화
    _configure_numpy_pandas()
    _configure_matplotlib()

    if seed is not None:
        set_seed(seed)

    elapsed = time.time() - t_start

    result = {
        **gpu_info,
        "optimal_num_workers": workers,
        "platform"           : platform.platform(),
        "python_version"     : sys.version.split()[0],
    }

    if verbose:
        print()
        print("  ┌─────────────────── 시스템 요약 ───────────────────┐")
        print(f"  │  OS       : {platform.platform()[:50]}")
        print(f"  │  Python   : {sys.version.split()[0]}")
        if "gpu_name" in result:
            print(f"  │  GPU      : {result['gpu_name']}")
            print(f"  │  VRAM     : {result['vram_total_gb']} GB")
            print(f"  │  CUDA     : {result.get('cuda_version','?')}  "
                  f"cuDNN: {result.get('cudnn_version','?')}")
        print(f"  │  Workers  : {workers}")
        print(f"  │  초기화   : {elapsed:.2f}초 완료")
        print("  └────────────────────────────────────────────────────┘")
        print()
        print("  📌 사용 가능한 유틸 함수:")
        print("     activate_cuda()            → CUDA 활성화 & 검증 (재실행)")
        print("     set_seed(42)               → 재현성 시드 고정")
        print("     gpu_memory_status()        → VRAM 사용량 확인")
        print("     clear_gpu_memory()         → GPU 캐시 비우기")
        print("     get_recommended_config()   → 작업별 권장 설정")
        print("     get_amp_scaler()           → FP16 GradScaler 반환")
        print()

    return result


# ─────────────────────────────────────────────────────────
# import 시 자동 실행
# ─────────────────────────────────────────────────────────
_config = setup()
device  = _config.get("device", None)      # 전역으로 바로 사용 가능


  ╔══════════════════════════════════════════════════╗
  ║      ML / DL 프로젝트 환경 초기화 중...         ║
  ║  i9-14900K | RTX 4070 12GB | CUDA 12.4          ║
  ╚══════════════════════════════════════════════════╝

  ✅ 경고 & 로깅 설정 완료
  ✅ CPU 멀티스레드 설정 완료 (threads=8, cpu_total=32)

  ┌─────────────── CUDA 활성화 & 검증 ────────────────┐
  │  ✅ CUDA 12.4  |  cuDNN 90100  |  Driver 591.74
  │  ✅ GPU   : NVIDIA GeForce RTX 4070
  │  ✅ CC    : 8.9  |  SM: 46개
  │  ✅ VRAM  : [██░░░░░░░░░░░░░░░░░░░░░░░] 1.16/11.99 GB (여유 10.83 GB)
  │  ✅ Context 초기화 완료
  │  ✅ Warm-up 완료 (CUDA / cuDNN / FP16)
  └────────────────────────────────────────────────────┘

  ✅ PyTorch 2.6.0+cu124 | CUDA 12.4 | cuDNN 90100
  ✅ GPU: NVIDIA GeForce RTX 4070 | VRAM: 11.99 GB | SM: 46 | CC: 8.9
  ✅ cuDNN benchmark=True | TF32=True | VRAM 단편화 방지 설정 완료
  ✅ NumPy 1.26.4 로드 완료
  ✅ Pandas 3.0.0 | Copy-on-Write=True 설정 완료
  ✅ Matplotlib 3.10.8 | Agg 백엔드 설정 완료

  ┌─────────────────── 시스템 요약 ───────────────────┐
  │  OS       : Windows-11-10

# 02. Data Acquisition (VitalDB 및 임상 데이터 로드)

In [2]:
# 임상 데이터 로드 및 사망/생존 현황 분석
import vitaldb
import pandas as pd

# 1. 임상 데이터 로드
df_clinical = pd.read_csv("https://api.vitaldb.net/cases")

# 2. 라벨 생성: 사망(0), 생존(1)
# 기존 death_inhosp 데이터(1:사망, 0:생존)를 모델 목적에 맞게 반전시킵니다.
df_clinical['survival_label'] = 1 - df_clinical['death_inhosp'] 

# 3. 타겟 변수 확인 (사망자 우선 순위)
counts = df_clinical['survival_label'].value_counts()
total_cases = len(df_clinical)

print(f"전체 데이터 수: {total_cases}건")
# 분석의 핵심인 사망자(Target)를 가장 먼저 노출합니다.
print(f"사망자 (0): {counts.get(0, 0)}명 | 생존자 (1): {counts.get(1, 0)}명")

# 4. 주요 지표 산출
mortality_rate = (counts.get(0, 0) / total_cases) * 100
survival_rate = (counts.get(1, 0) / total_cases) * 100
print(f"수술 중 생존율 : {survival_rate:.2f}%")
print(f"수술 중 사망률 : {mortality_rate:.2f}%")

전체 데이터 수: 6388건
사망자 (0): 57명 | 생존자 (1): 6331명
수술 중 생존율 : 99.11%
수술 중 사망률 : 0.89%


In [3]:
import vitaldb
import pandas as pd
import numpy as np
import warnings
from joblib import Parallel, delayed
from tqdm import tqdm

warnings.filterwarnings('ignore')

# 설정값
SAVE_PATH = "./vitaldb_mortality_medical.npz" 
INTERVAL = 5      
MAX_LEN = 3600    
MIN_LEN = 360     
NUM_CORES = 24    

# 데이터 트랙 설정 (한글 주석 추가)
TRACKS = [
    'Solar8000/HR',          # 심박수 (Heart Rate)
    'Solar8000/NIBP_MBP',     # 평균 혈압 (Mean Blood Pressure)
    'Solar8000/PLETH_SPO2',   # 맥박 산소 포화도 (Oxygen Saturation)
    'Solar8000/ST_II',        # 심전도 ST분절 (ST Segment)
    'Solar8000/RR',           # 호흡수 (Respiration Rate)
    'Solar8000/EtCO2',        # 호흡말 이산화탄소 (End-tidal CO2)
    'Solar8000/BT',           # 체온 (Body Temperature)
    'BIS/BIS'                # 마취 깊이 지수 (Bispectral Index)
]

def clean_vital_data(df):
    """임상적 허용 범위를 벗어난 이상치 제거 및 결측치 보간"""
    mask_hr = (df['HR'] < 20) | (df['HR'] > 250)
    mask_mbp = (df['MBP'] < 20) | (df['MBP'] > 300)
    mask_spo2 = (df['SPO2'] < 20)
    
    df.loc[mask_hr, 'HR'] = np.nan
    df.loc[mask_mbp, 'MBP'] = np.nan
    df.loc[mask_spo2, 'SPO2'] = np.nan
    
    df = df.interpolate(method='linear', limit_direction='both')
    df = df.ffill().bfill().fillna(0)
    return df

def process_case(caseid, row_data, median_stats):
    """개별 케이스의 Vital 데이터 및 정적 데이터 전처리"""
    try:
        vals = vitaldb.load_case(caseid, TRACKS, interval=INTERVAL)
        if vals is None or len(vals) < MIN_LEN:
            return None

        cols = ['HR', 'MBP', 'SPO2', 'ST', 'RR', 'CO2', 'BT', 'BIS']
        case_df = pd.DataFrame(vals, columns=cols)
        case_df = clean_vital_data(case_df)
        data_np = case_df.to_numpy().astype(np.float32)

        # Sequence 길이 조정
        if len(data_np) > MAX_LEN:
            data_np = data_np[:MAX_LEN]
        else:
            padding = np.zeros((MAX_LEN - len(data_np), len(TRACKS)), dtype=np.float32)
            data_np = np.vstack([data_np, padding])

        # 정적 변수 결측치 처리 (중앙값 활용)
        age = row_data.get('age', median_stats['age'])
        height = row_data.get('height', median_stats['height'])
        weight = row_data.get('weight', median_stats['weight'])
        bmi = row_data.get('bmi', median_stats['bmi'])
        
        static_features = np.array([
            age, 1 if row_data.get('sex') == 'M' else 0,
            height, weight, bmi, row_data.get('asa', 2)
        ], dtype=np.float32)
        
        # 라벨 반전 (death_inhosp 1 -> 사망 0, 0 -> 생존 1)
        label = 1 - int(row_data['death_inhosp'])

        return {'dynamic': data_np, 'static': static_features, 'label': label}
    except Exception:
        return None

if __name__ == '__main__':
    print("VitalDB 데이터 구축 작업을 시작합니다.")
    
    df_clinical = pd.read_csv("https://api.vitaldb.net/cases")
    median_stats = df_clinical[['age', 'height', 'weight', 'bmi']].median().to_dict()
    case_list = df_clinical.to_dict('records')
    
    # 병렬 처리 실행
    results = Parallel(n_jobs=NUM_CORES, backend="threading")(
        delayed(process_case)(row['caseid'], row, median_stats) 
        for row in tqdm(case_list, desc="데이터 로드 및 전처리")
    )
    
    valid_results = [r for r in results if r is not None]
    
    X_dynamic = np.array([r['dynamic'] for r in valid_results])
    X_static = np.array([r['static'] for r in valid_results])
    y = np.array([r['label'] for r in valid_results])
    
    # NPZ 압축 저장
    np.savez_compressed(SAVE_PATH, X_dynamic=X_dynamic, X_static=X_static, y=y)
    
    # 통계 계산
    survivors = sum(y)
    deaths = len(y) - survivors

    # 구축 완료 보고
    print("\n" + "="*50)
    print("데이터셋 구축 완료")
    print(f" - 전체 샘플 수     : {len(y)}개")
    print(f" - 동적 데이터 Shape : {X_dynamic.shape}")
    print(f" - 사망자 수 (0)    : {deaths}명")
    print(f" - 생존자 수 (1)    : {survivors}명")
    print(f" - 라벨 적용 수식   : Label = 1 - death_inhosp")
    print("="*50)

VitalDB 데이터 구축 작업을 시작합니다.


데이터 로드 및 전처리: 100%|██████████| 6388/6388 [04:29<00:00, 23.70it/s]



데이터셋 구축 완료
 - 전체 샘플 수     : 6385개
 - 동적 데이터 Shape : (6385, 3600, 8)
 - 사망자 수 (0)    : 57명
 - 생존자 수 (1)    : 6328명
 - 라벨 적용 수식   : Label = 1 - death_inhosp


# 03. Medical Grade Data Preprocessing (의학적 전처리 및 데이터셋 구축)

In [4]:
import vitaldb
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────
# 1. 하이엔드 시스템 및 데이터 규격 설정
# ─────────────────────────────────────────────────────────
INTERVAL = 5      # 5초 간격 (초정밀 해상도)
MAX_LEN = 3600    # 최대 5시간 시퀀스 길이
MIN_LEN = 360     # 최소 30분 이상의 유효 데이터 확보
NUM_CORES = 24    # i9-14900K 풀코어(24 Cores) 활용

# 8대 골든아워 지표
TRACKS = [
    'Solar8000/HR',         # 심박수 (Heart Rate)
    'Solar8000/NIBP_MBP',   # 평균 동맥압 (Mean Blood Pressure)
    'Solar8000/PLETH_SPO2', # 맥박 산소포화도 (Oxygen Saturation)
    'Solar8000/ST_II',      # 심전도 ST 분절 (ST Segment - 심근 허혈 감지)
    'Solar8000/RR',         # 호흡수 (Respiratory Rate)
    'Solar8000/EtCO2',      # 호기말 이산화탄소 분압 (End-tidal CO2)
    'Solar8000/BT',         # 체온 (Body Temperature)
    'BIS/BIS'               # 마취 심도 지수 (Bispectral Index)
]

# ─────────────────────────────────────────────────────────
# 2. 의학적 데이터 정제 함수 (Medical Refinement)
# ─────────────────────────────────────────────────────────
def clean_vital_data(df):
    """
    생리학적 이상치를 제거하고 시계열 연속성을 확보하기 위한 전처리를 수행합니다.
    """
    # 1. 이상치 탐지 및 제거 (Medical Standard)
    mask_hr = (df['HR'] < 20) | (df['HR'] > 250)      # 심박수 이상치
    mask_mbp = (df['MBP'] < 20) | (df['MBP'] > 300)   # 평균 혈압 이상치
    mask_spo2 = (df['SPO2'] < 20)                     # 산소포화도 센서 오류
    
    df.loc[mask_hr, 'HR'] = np.nan
    df.loc[mask_mbp, 'MBP'] = np.nan
    df.loc[mask_spo2, 'SPO2'] = np.nan
    
    # 2. 선형 보간 (Linear Interpolation): 끊긴 신호를 직선으로 연결
    df = df.interpolate(method='linear', limit_direction='both')
    
    # 3. 보간 후 남은 결측치 방어 처리 (앞뒤 값 채우기)
    df = df.ffill().bfill().fillna(0)
    return df

# ─────────────────────────────────────────────────────────
# 3. 개별 케이스 프로세싱 엔진 (Parallel Engine)
# ─────────────────────────────────────────────────────────
def process_case(caseid, row_data, median_stats):
    try:
        # 98GB 데이터셋으로부터 실시간 생체 신호 로드
        vals = vitaldb.load_case(caseid, TRACKS, interval=INTERVAL)
        if vals is None or len(vals) < MIN_LEN:
            return None

        # [A] 동적 시퀀스 데이터 처리 (Dynamic Features)
        # 매핑 순서 : 심박수, 혈압, 산소포화도, ST분절, 호흡수, CO2, 체온, 마취심도
        cols = ['HR', 'MBP', 'SPO2', 'ST', 'RR', 'CO2', 'BT', 'BIS']
        case_df = pd.DataFrame(vals, columns=cols)
        case_df = clean_vital_data(case_df)
        data_np = case_df.to_numpy().astype(np.float32)

        # 길이 정규화 (Padding & Truncating)
        if len(data_np) > MAX_LEN:
            data_np = data_np[:MAX_LEN]
        else:
            padding = np.zeros((MAX_LEN - len(data_np), len(TRACKS)), dtype=np.float32)
            data_np = np.vstack([data_np, padding])

        # [B] 정적 데이터 처리 (Static Features)
        # 나이, 성별, 키, 몸무게, BMI, ASA등급
        static_features = np.array([
            row_data.get('age', median_stats['age']),
            1 if row_data.get('sex') == 'M' else 0,
            row_data.get('height', median_stats['height']),
            row_data.get('weight', median_stats['weight']),
            row_data.get('bmi', median_stats['bmi']),
            row_data.get('asa', 2) # ASA 등급 결측 시 2(보통) 적용
        ], dtype=np.float32)
        
        # [C] 레이블 변환: 사망 = 0, 생존 = 1
        label = 1 - int(row_data['death_inhosp'])

        return {'dynamic': data_np, 'static': static_features, 'label': label}
    except Exception:
        return None

# ─────────────────────────────────────────────────────────
# 4. 메모리 집약적 병렬 실행 (128GB RAM 활용)
# ─────────────────────────────────────────────────────────
if __name__ == '__main__':
    print(f"🚀 [Project] Medical Grade Data Pipeline Ready")
    print(f" -> Workstation: i9-14900K | 128GB RAM | 24 Multi-Cores")
    
    # 기저 데이터 및 중앙값 통계 로드
    df_clinical = pd.read_csv("https://api.vitaldb.net/cases")
    median_stats = df_clinical[['age', 'height', 'weight', 'bmi']].median().to_dict()
    
    # i9-14900K 24코어 병렬 처리 실행
    results = Parallel(n_jobs=NUM_CORES, backend="threading")(
        delayed(process_case)(row['caseid'], row, median_stats) 
        for row in tqdm(df_clinical.to_dict('records'), desc="Processing Bio-Signals")
    )
    
    # 128GB RAM을 활용하여 데이터를 메모리에 상주
    valid_results = [r for r in results if r is not None]
    
    X_dynamic = np.array([r['dynamic'] for r in valid_results])
    X_static = np.array([r['static'] for r in valid_results])
    y = np.array([r['label'] for r in valid_results])
    
    # 최종 통합 결과 요약
    survivors, deaths = sum(y), len(y) - sum(y)
    print("\n" + "="*60)
    print(f"✅ Preprocessing Complete : Data ready for EDA/Modeling")
    print(f" - X_dynamic (시계열 파형) : {X_dynamic.shape}")
    print(f" - X_static (환자 메타) : {X_static.shape}")
    print(f" - y (레이블): 사망(0) : {deaths}명 | 생존(1) : {survivors}명")
    print("="*60)

🚀 [Project] Medical Grade Data Pipeline Ready
 -> Workstation: i9-14900K | 128GB RAM | 24 Multi-Cores


Processing Bio-Signals: 100%|██████████| 6388/6388 [07:00<00:00, 15.19it/s]



✅ Preprocessing Complete : Data ready for EDA/Modeling
 - X_dynamic (시계열 파형) : (6385, 3600, 8)
 - X_static (환자 메타) : (6385, 6)
 - y (레이블): 사망(0) : 57명 | 생존(1) : 6328명


# 04. Exploratory Data Analysis (EDA) : 데이터 분포 및 정규성 검정

In [5]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pandas as pd
import numpy as np

# 1. 시각화용 데이터프레임 생성
# 컬럼: [나이, 성별, 키, 몸무게, BMI, ASA등급]
static_cols = ['Age', 'Sex', 'Height', 'Weight', 'BMI', 'ASA']
df_eda = pd.DataFrame(X_static, columns=static_cols)
df_eda['Label'] = y  # 0 : 사망, 1 : 생존

# 2. 그룹별 주요 변수 분포 시각화 (Age, BMI, ASA)
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# [A] 나이 분포: 사망자 그룹의 고령화 여부 확인
sns.boxplot(x='Label', y='Age', data=df_eda, ax=axes[0], palette='coolwarm')
axes[0].set_title('Age Distribution by Survival (0:Dead, 1:Surv)', fontsize=13)

# [B] BMI 분포: 체질량 지수와 사망의 상관관계
sns.boxplot(x='Label', y='BMI', data=df_eda, ax=axes[1], palette='viridis')
axes[1].set_title('BMI Distribution by Survival', fontsize=13)

# [C] ASA 등급: 환자의 전신 상태 점수와 사망 관계
sns.countplot(x='ASA', hue='Label', data=df_eda, ax=axes[2])
axes[2].set_title('ASA Physical Status Grade', fontsize=13)
axes[2].set_yscale('log') # 불균형이 심하므로 로그 스케일 적용

plt.tight_layout()
plt.show()

# 3. 정규성 검정 (Normality Test)
print("📌 수치형 변수 정규성 및 통계 요약")
for col in ['Age', 'Height', 'Weight', 'BMI']:
    skewness = df_eda[col].skew()
    kurtosis = df_eda[col].kurt()
    # Shapiro-Wilk 검정은 표본이 너무 크면(>5000) 민감하므로 왜도/첨도 위주로 판단
    print(f" - {col:<7}: 왜도(Skewness) = {skewness:.2f}, 첨도(Kurtosis) = {kurtosis:.2f}")

# 4. 피처 간 상관관계 분석
plt.figure(figsize=(10, 8))
sns.heatmap(df_eda.corr(), annot=True, cmap='RdBu_r', fmt='.2f', linewidths=0.5)
plt.title("Clinical Features Correlation Matrix", fontsize=15)
plt.show()

📌 수치형 변수 정규성 및 통계 요약
 - Age    : 왜도(Skewness) = -0.60, 첨도(Kurtosis) = 0.26
 - Height : 왜도(Skewness) = -2.18, 첨도(Kurtosis) = 21.36
 - Weight : 왜도(Skewness) = 0.37, 첨도(Kurtosis) = 1.78
 - BMI    : 왜도(Skewness) = 0.53, 첨도(Kurtosis) = 1.23


# 05. Data Scaling & Model Input Preparation (데이터 스케일링)

In [6]:
import copy
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

# 1. 스케일러 초기화 (중앙값 기반으로 이상치 영향 최소화)
scaler_static = RobustScaler()
scaler_dynamic = RobustScaler()

# 2. 정적 변수 스케일링 (X_static)
# 원본 데이터 보호를 위해 deepcopy 수행
X_static_scaled = copy.deepcopy(X_static)

# [Age(0), Height(2), Weight(3), BMI(4)] 인덱스만 선택하여 스케일링
cols_to_scale = [0, 2, 3, 4]
X_static_scaled[:, cols_to_scale] = scaler_static.fit_transform(X_static[:, cols_to_scale])

# 3. 시계열 데이터 스케일링 (X_dynamic)
# 원본 시계열 데이터 복사
X_dynamic_scaled = copy.deepcopy(X_dynamic)

# (환자수, 시간, 지표수) 구조에서 각 지표(채널)별로 독립적 스케일링 수행
# 128GB RAM의 넓은 대역폭을 활용하여 대량의 텐서 연산 처리
N, T, C = X_dynamic_scaled.shape
for i in range(C):
    # 채널별 데이터를 추출하여 스케일링 후 다시 원래 모양으로 복원
    channel_data = X_dynamic_scaled[:, :, i].reshape(-1, 1)
    X_dynamic_scaled[:, :, i] = scaler_dynamic.fit_transform(channel_data).reshape(N, T)

print("✅ 데이터 스케일링 완료 (copy.deepcopy 적용)")
print(f" - 스케일링 전 Height 첨도 : 21.36")
print(f" - 스케일링 후 Height 첨도 : {pd.Series(X_static_scaled[:, 2]).kurt():.2f}")
print(f" - 데이터 준비 상태 : {X_dynamic_scaled.shape}, {X_static_scaled.shape}")

✅ 데이터 스케일링 완료 (copy.deepcopy 적용)
 - 스케일링 전 Height 첨도 : 21.36
 - 스케일링 후 Height 첨도 : 21.36
 - 데이터 준비 상태 : (6385, 3600, 8), (6385, 6)


# 06. Balanced Data Supply Chain (불균형 데이터 최적화 및 공급)

In [7]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from sklearn.model_selection import train_test_split

# 1. 시계열 추세 피처 추출 함수 정의
def extract_clinical_trends(dynamic_data) :
    batch_size, seq_len, num_features = dynamic_data.shape
    # 최근 10분(600개 시퀀스) 데이터 분석
    recent_data = dynamic_data[:, -600 :, :]
    
    mean_vals = np.mean(recent_data, axis = 1)
    std_vals = np.std(recent_data, axis = 1)
    # 변동성 및 기울기 계산
    cv_vals = std_vals / (mean_vals + 1e-8)
    slopes = recent_data[:, -1, :] - recent_data[:, 0, :]
    
    trend_features = np.hstack([mean_vals, std_vals, cv_vals, slopes])
    return trend_features.astype(np.float32)

# 2. 데이터셋 클래스 정의
class VitalDataset(Dataset) :
    def __init__(self, dynamic, static, labels) :
        self.dynamic = torch.FloatTensor(dynamic)
        self.static = torch.FloatTensor(static)
        self.labels = torch.FloatTensor(labels).unsqueeze(1)

    def __len__(self) :
        return len(self.labels)

    def __getitem__(self, idx) :
        return self.dynamic[idx], self.static[idx], self.labels[idx]

# 3. 추세 피처 생성 및 정적 데이터 결합
# 이 단계에서 X_static_combined 변수가 생성됩니다.
X_trend = extract_clinical_trends(X_dynamic_scaled)
X_static_combined = np.hstack([X_static_scaled, X_trend])

# 4. 데이터셋 인스턴스 생성 (38개 피처 반영)
full_dataset = VitalDataset(X_dynamic_scaled, X_static_combined, y)
indices = np.arange(len(full_dataset))

train_idx, val_idx = train_test_split(
    indices, 
    test_size = 0.2, 
    stratify = y, 
    random_state = 42
)

# 5. 클래스 불균형 해소를 위한 샘플러 설정
y_train = y[train_idx]
class_sample_count = np.array([len(np.where(y_train == t)[0]) for t in np.unique(y_train)])
weight = 1. / class_sample_count
samples_weight = np.array([weight[int(t)] for t in y_train])
samples_weight = torch.from_numpy(samples_weight).double()

sampler = WeightedRandomSampler(samples_weight, len(samples_weight))

# 6. 데이터 로더 구성
BATCH_SIZE = 16

train_loader = DataLoader(
    Subset(full_dataset, train_idx), 
    batch_size = BATCH_SIZE, 
    sampler = sampler, 
    num_workers = 0, 
    pin_memory = True, 
    drop_last = True
)

val_loader = DataLoader(
    Subset(full_dataset, val_idx), 
    batch_size = BATCH_SIZE, 
    shuffle = False, 
    pin_memory = True
)

print("-" * 30)
print(f"추세 피처 생성 완료 (Total Features : 38)")
print(f"학습 배치 수 : {len(train_loader)}")
print(f"검증 배치 수 : {len(val_loader)}")
print(f"샘플링 전략 : WeightedRandomSampler 적용")
print("-" * 30)

------------------------------
추세 피처 생성 완료 (Total Features : 38)
학습 배치 수 : 319
검증 배치 수 : 80
샘플링 전략 : WeightedRandomSampler 적용
------------------------------


# 07. Medical Transformer Architecture

In [8]:
import torch
import torch.nn as nn

class MedicalTransformer(nn.Module) :
    def __init__(self, dynamic_dim = 8, static_dim = 38, hidden_dim = 128, nhead = 8, num_layers = 3) :
        super(MedicalTransformer, self).__init__()
        
        self.dynamic_projection = nn.Linear(dynamic_dim, hidden_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model = hidden_dim, 
            nhead = nhead, 
            dim_feedforward = hidden_dim * 4,
            dropout = 0.3, # 시계열 파트도 0.3으로 약간 상향
            batch_first = True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers = num_layers)
        
        self.static_fc = nn.Sequential(
            nn.Linear(static_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.4), # 오버피팅 억제
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.4) # 오버피팅 억제
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.4), # 오버피팅 억제
            nn.Linear(64, 1)
        )

    def forward(self, dynamic_input, static_input) :
        dyn_feat = self.dynamic_projection(dynamic_input)
        dyn_feat = self.transformer_encoder(dyn_feat)
        
        dyn_feat = torch.mean(dyn_feat, dim = 1)
        stat_feat = self.static_fc(static_input)
        
        combined = torch.cat([dyn_feat, stat_feat], dim = 1)
        logits = self.classifier(combined)
        return logits

model = MedicalTransformer(static_dim = 38).to(device)

# 08. Model Optimization Setup (Focal Loss & amsgrad)

In [9]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import GradScaler

class FocalLoss(nn.Module) :
    def __init__(self, alpha = 0.95, gamma = 4.0) : # gamma를 4.0으로 높여 아주 어려운 샘플에 초집중
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets) :
        ce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction = 'none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt)**self.gamma * ce_loss
        return focal_loss.mean()

criterion = FocalLoss(alpha = 0.95, gamma = 4.0)
# weight_decay를 0.2로 높여 파라미터가 비정상적으로 커지는 것을 강하게 억제
optimizer = optim.AdamW(model.parameters(), lr = 3e-5, weight_decay = 0.2, amsgrad = True)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode = 'min', factor = 0.5, patience = 2, min_lr = 1e-7)
scaler = GradScaler()

# 09. Model Training & Monitoring (Mixed Precision & Accumulation)

In [10]:
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score
from torch.amp import autocast

epochs = 50
best_auprc = 0.0
accumulation_steps = 4 

for epoch in range(epochs) :
    model.train()
    train_loss = 0.0
    train_pbar = tqdm(train_loader, desc = f"Epoch {epoch + 1}/{epochs} [Train]")
    
    optimizer.zero_grad(set_to_none = True)
    
    for i, (dyn, stat, labels) in enumerate(train_pbar) :
        dyn, stat, labels = dyn.to(device), stat.to(device), labels.to(device)
        
        # Mixed Precision을 통한 RTX 4070 가속
        with autocast(device_type = 'cuda') :
            outputs = model(dyn, stat)
            loss = criterion(outputs, labels) / accumulation_steps
            
        if not torch.isnan(loss) :
            scaler.scale(loss).backward()
            
            # Gradient Accumulation (실질 배치 64 운영)
            if (i + 1) % accumulation_steps == 0 :
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none = True)
            
            train_loss += loss.item() * accumulation_steps
            train_pbar.set_postfix({'loss' : f'{loss.item() * accumulation_steps : .4f}'})
        
    # 검증 단계
    model.eval()
    val_preds, val_labels = [], []
    current_val_loss = 0.0
    valid_batches = 0
    val_pbar = tqdm(val_loader, desc = f"Epoch {epoch + 1}/{epochs} [Val]", leave = False)
    
    with torch.no_grad() :
        for dyn, stat, labels in val_pbar :
            dyn, stat, labels = dyn.to(device), stat.to(device), labels.to(device)
            
            with autocast(device_type = 'cuda') :
                # 수치 안정성을 위해 clamp 적용
                outputs = torch.clamp(model(dyn, stat), min = -15.0, max = 15.0)
                v_loss = criterion(outputs, labels)
            
            if not torch.isnan(v_loss) :
                current_val_loss += v_loss.item()
                valid_batches += 1
            
            probs = torch.sigmoid(outputs)
            val_preds.extend(torch.nan_to_num(probs, nan = 0.0).cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
            
    val_preds_np = np.array(val_preds).flatten()
    val_labels_np = np.array(val_labels).flatten()
    avg_val_loss = current_val_loss / valid_batches if valid_batches > 0 else 0.0
    
    try :
        val_auroc = roc_auc_score(val_labels_np, val_preds_np)
        val_auprc = average_precision_score(val_labels_np, val_preds_np)
    except :
        val_auroc, val_auprc = 0.5, 0.0
    
    # 최고의 AUPRC를 보이는 모델 저장
    if val_auprc > best_auprc :
        best_auprc = val_auprc
        torch.save(model.state_dict(), "best_medical_transformer.pth")

    scheduler.step(avg_val_loss)
    
    print(f"결과 요약 - Loss (T/V) : {train_loss / len(train_loader) : .4f} / {avg_val_loss : .4f} | "
          f"AUROC : {val_auroc : .4f} | AUPRC : {val_auprc : .4f}")

Epoch 1/50 [Train]:   0%|          | 0/319 [00:00<?, ?it/s]

Epoch 1/50 [Val]:   0%|          | 0/80 [00:00<?, ?it/s]

결과 요약 - Loss (T/V) :  0.0319 /  0.0411 | AUROC :  0.7166 | AUPRC :  0.9957


Epoch 2/50 [Train]:   0%|          | 0/319 [00:00<?, ?it/s]

Epoch 2/50 [Val]:   0%|          | 0/80 [00:00<?, ?it/s]

...

Epoch 49/50 [Val]:   0%|          | 0/80 [00:00<?, ?it/s]

결과 요약 - Loss (T/V) :  0.0179 /  0.0257 | AUROC :  0.5479 | AUPRC :  0.9920


Epoch 50/50 [Train]:   0%|          | 0/319 [00:00<?, ?it/s]

Epoch 50/50 [Val]:   0%|          | 0/80 [00:00<?, ?it/s]

결과 요약 - Loss (T/V) :  0.0180 /  0.0257 | AUROC :  0.5484 | AUPRC :  0.9920


# 10. Model Diagnostics & Clinical Performance Report (모델 진단 및 임상 성능 보고서)

In [11]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, confusion_matrix, classification_report
import torch
from torch.amp import autocast

# 최고 성능 모델(AUPRC 기준) 로드
model.load_state_dict(torch.load("best_medical_transformer.pth", weights_only = True))
model.eval()

y_true_list, y_probs_list = [], []

with torch.no_grad() :
    for dyn, stat, labels in val_loader :
        dyn, stat, labels = dyn.to(device), stat.to(device), labels.to(device)
        
        # 평가 시에도 Mixed Precision 및 수치 안정화 로직 적용
        with autocast(device_type = 'cuda') :
            outputs = model(dyn, stat)
            outputs = torch.clamp(outputs, min = -15.0, max = 15.0)
            probs = torch.sigmoid(outputs)
        
        y_probs_list.extend(torch.nan_to_num(probs, nan = 0.0).cpu().numpy())
        y_true_list.extend(labels.cpu().numpy())

y_true_arr = np.array(y_true_list).flatten()
y_probs_arr = np.array(y_probs_list).flatten()

# 사망(0)을 탐지 대상(Positive)으로 설정하기 위해 레이블 반전
# 원본 데이터 : 0 = 사망, 1 = 생존 -> 분석용 : 1 = 사망, 0 = 생존
y_true_death = 1 - y_true_arr
y_probs_death = 1 - y_probs_arr

# F2-Score 기준 최적 임계값(Threshold) 탐색
# 임상에서는 사망자를 놓치지 않는 것(Recall)이 정밀도(Precision)보다 중요하므로 F2-Score를 사용합니다.
precision, recall, thresholds = precision_recall_curve(y_true_death, y_probs_death)
f2_scores = (1 + 2**2) * (precision * recall) / ((2**2 * precision) + recall + 1e-8)
best_idx = np.argmax(f2_scores)
opt_threshold = thresholds[best_idx] if best_idx < len(thresholds) else thresholds[-1]

# 최적 임계값 기반 예측 및 혼동 행렬(Confusion Matrix) 생성
y_preds_opt = (y_probs_death >= opt_threshold).astype(int)
cm = confusion_matrix(y_true_death, y_preds_opt)

# 시각화 보고서 출력
plt.figure(figsize = (7, 6))
sns.heatmap(
    cm, 
    annot = True, 
    fmt = 'd', 
    cmap = 'YlOrRd', 
    xticklabels = ['Predicted : Normal', 'Predicted : High-Risk'], 
    yticklabels = ['Actual : Survive', 'Actual : Death']
)
plt.title(f'Clinical Risk Matrix (38 Features Integration)\nOptimized Threshold : {opt_threshold : .6f}')
plt.ylabel('Clinical Reality')
plt.xlabel('Model Prediction')
plt.show()

# 최종 임상 지표 리포트 출력
print(f"📊 [Clinical Performance Report]")
print("-" * 50)
print(f"✅ 최적 위험 판단 임계값 : {opt_threshold : .6f}")
print(f"✅ 사망자 탐지 재현율 (Recall) : {recall[best_idx] : .4f}")
print(f"✅ 위험 예측 정밀도 (Precision) : {precision[best_idx] : .4f}")
print(f"✅ 분석 대상 환자 수 : {len(y_probs_arr)} 명")
print("-" * 50)
print(classification_report(y_true_death, y_preds_opt, target_names = ['Normal', 'High-Risk']))

📊 [Clinical Performance Report]
--------------------------------------------------
✅ 최적 위험 판단 임계값 :  0.571289
✅ 사망자 탐지 재현율 (Recall) :  0.2727
✅ 위험 예측 정밀도 (Precision) :  0.1034
✅ 분석 대상 환자 수 : 1277 명
--------------------------------------------------
              precision    recall  f1-score   support

      Normal       0.99      0.98      0.99      1266
   High-Risk       0.10      0.27      0.15        11

    accuracy                           0.97      1277
   macro avg       0.55      0.63      0.57      1277
weighted avg       0.99      0.97      0.98      1277



# 11. Threshold Optimization & Error Diagnostics

In [12]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.amp import autocast
from sklearn.metrics import precision_recall_curve, roc_curve, auc, classification_report

# 1. Inference & Data Collection
model.load_state_dict(torch.load("best_medical_transformer.pth", weights_only = True))
model.eval()

y_true_list, y_probs_list = [], []

with torch.no_grad() :
    for dyn, stat, labels in val_loader :
        dyn, stat, labels = dyn.to(device), stat.to(device), labels.to(device)
        with autocast(device_type = 'cuda') :
            outputs = torch.clamp(model(dyn, stat), min = -15.0, max = 15.0)
            probs = torch.sigmoid(outputs)
        y_probs_list.extend(torch.nan_to_num(probs, nan = 0.0).cpu().numpy())
        y_true_list.extend(labels.cpu().numpy())

y_true = 1 - np.array(y_true_list).flatten()
y_probs = 1 - np.array(y_probs_list).flatten()

# 2. Metric Calculation
fpr, tpr, _ = roc_curve(y_true, y_probs)
precision, recall, pr_thresholds = precision_recall_curve(y_true, y_probs)
roc_auc = auc(fpr, tpr)
pr_auc = auc(recall, precision)

# 3. Diagnostic Visualization

plt.figure(figsize = (14, 6))

plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, color = 'crimson', lw = 2, label = f'ROC AUC = {roc_auc : .3f}')
plt.plot([0, 1], [0, 1], color = 'gray', linestyle = '--')
plt.title('Receiver Operating Characteristic (ROC)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc = "lower right")
plt.grid(alpha = 0.3)

plt.subplot(1, 2, 2)
plt.plot(recall, precision, color = 'teal', lw = 2, label = f'PR AUC = {pr_auc : .3f}')
plt.title('Precision-Recall (PR) Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend(loc = "upper right")
plt.grid(alpha = 0.3)

plt.tight_layout()
plt.show()

# 4. Final Performance Report
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
best_idx = np.argmax(f1_scores)
opt_threshold = pr_thresholds[best_idx] if best_idx < len(pr_thresholds) else pr_thresholds[-1]

y_preds = (y_probs >= opt_threshold).astype(int)

print(f"✅ 분석 결과 리포트")
print("-" * 50)
print(f"최적 임계값 (Best Threshold) : {opt_threshold : .6f}")
print(f"사망자 포착률 (Recall) : {recall[best_idx] : .4f}")
print(f"예측 정확도 (Precision) : {precision[best_idx] : .4f}")
print("-" * 50)
print(classification_report(y_true, y_preds, target_names = ['Normal', 'High-Risk']))

✅ 분석 결과 리포트
--------------------------------------------------
최적 임계값 (Best Threshold) :  0.571289
사망자 포착률 (Recall) :  0.2727
예측 정확도 (Precision) :  0.1034
--------------------------------------------------
              precision    recall  f1-score   support

      Normal       0.99      0.98      0.99      1266
   High-Risk       0.10      0.27      0.15        11

    accuracy                           0.97      1277
   macro avg       0.55      0.63      0.57      1277
weighted avg       0.99      0.97      0.98      1277



# 12. Model Diagnostics: High-Risk Detection (사망 위험군 정밀 분석)

In [13]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, confusion_matrix, classification_report

# 1. 데이터 정제 및 변환 (NaN 완벽 방어)
y_true_arr = np.array(y_true).flatten()
y_probs_arr = np.array(y_probs).flatten()

# 결측치(NaN)가 있는 인덱스 찾아내서 동시 제거
valid_mask = ~np.isnan(y_probs_arr)
y_true_clean = y_true_arr[valid_mask]
y_probs_clean = y_probs_arr[valid_mask]

# 사망(0)을 Positive(1)로 취급하기 위해 반전
y_true_death = 1 - y_true_clean
y_probs_death = 1 - y_probs_clean

# 2. 사망자 타겟 PR 곡선 계산
precision, recall, thresholds = precision_recall_curve(y_true_death, y_probs_death)

# 3. F-beta Score를 이용한 최적 임계값 탐색
# 의료 데이터는 Recall에 더 무게를 두어야 하므로 beta=2 적용
beta = 2.0
f2_scores = (1 + beta**2) * (precision * recall) / ((beta**2 * precision) + recall + 1e-8)
best_idx = np.argmax(f2_scores)
opt_threshold = thresholds[best_idx] if best_idx < len(thresholds) else thresholds[-1]

# 4. 시각화: 오차 행렬 및 성능 곡선
plt.figure(figsize=(12, 5))

# Left: Confusion Matrix (위험군 탐지 결과)
plt.subplot(1, 2, 1)
y_preds_opt = (y_probs_death >= opt_threshold).astype(int)
cm = confusion_matrix(y_true_death, y_preds_opt)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', 
            xticklabels=['Safe', 'High-Risk'], 
            yticklabels=['Survive', 'Death'])
plt.title(f'Risk Detection Matrix\n(Threshold: {opt_threshold:.4f})')
plt.xlabel('Predicted Status')
plt.ylabel('Actual Status')

# Right: P-R Curve (최적점 표시)
plt.subplot(1, 2, 2)
plt.plot(recall, precision, 'b-', label='P-R Curve')
plt.plot(recall[best_idx], precision[best_idx], 'ro', label='Optimal Point (F2)')
plt.xlabel('Recall (Detection Rate)')
plt.ylabel('Precision (Accuracy)')
plt.title('High-Risk Detection Trade-off')
plt.legend()

plt.tight_layout()
plt.show()

# 5. 최종 리포트 출력
print(f"📊 [Clinical Report: Death Risk Prediction]")
print("-" * 45)
print(f"✅ 최적 판단 임계값(Threshold): {opt_threshold:.6f}")
print(f"✅ 사망자 포착률(Recall)      : {recall[best_idx]:.4f}")
print(f"✅ 위험군 예측 정확도(Precision) : {precision[best_idx]:.4f}")
print("-" * 45)
print(classification_report(y_true_death, y_preds_opt, target_names=['Normal', 'High-Risk']))

📊 [Clinical Report: Death Risk Prediction]
---------------------------------------------
✅ 최적 판단 임계값(Threshold): 0.000000
✅ 사망자 포착률(Recall)      : 1.0000
✅ 위험군 예측 정확도(Precision) : 0.9914
---------------------------------------------
              precision    recall  f1-score   support

      Normal       0.00      0.00      0.00        11
   High-Risk       0.99      1.00      1.00      1266

    accuracy                           0.99      1277
   macro avg       0.50      0.50      0.50      1277
weighted avg       0.98      0.99      0.99      1277



# 13. Model Diagnostics (사망 위험군 정밀 진단)

In [14]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, confusion_matrix, classification_report
import torch
from torch.amp import autocast

# 최고 성능 모델 로드
model.load_state_dict(torch.load("best_medical_transformer.pth", weights_only = True))
model.eval()

y_true_list, y_probs_list = [], []

with torch.no_grad():
    for dyn, stat, labels in val_loader:
        dyn, stat, labels = dyn.to(device), stat.to(device), labels.to(device)
        
        # 평가 단계에도 Mixed Precision 및 Clamp 방어 로직 적용
        with autocast(device_type = 'cuda'):
            outputs = model(dyn, stat)
            outputs = torch.clamp(outputs, min = -15.0, max = 15.0)
            probs = torch.sigmoid(outputs)
        
        # 만약의 nan 값도 0.0으로 안전하게 치환하여 유실 방지
        y_probs_list.extend(torch.nan_to_num(probs, nan = 0.0).cpu().numpy())
        y_true_list.extend(labels.cpu().numpy())

# 넘파이 배열 변환
y_true_arr = np.array(y_true_list).flatten()
y_probs_arr = np.array(y_probs_list).flatten()

# 사망(0)을 Positive(1)로 인식하도록 데이터 반전
y_true_death = 1 - y_true_arr
y_probs_death = 1 - y_probs_arr

# F2-Score(Recall 가중) 기준 최적 임계값 탐색
precision, recall, thresholds = precision_recall_curve(y_true_death, y_probs_death)
f2_scores = (1 + 2**2) * (precision * recall) / ((2**2 * precision) + recall + 1e-8)
best_idx = np.argmax(f2_scores)

opt_threshold = thresholds[-1] if best_idx >= len(thresholds) else thresholds[best_idx]

# 최종 진단 결과 시각화
y_preds_opt = (y_probs_death >= opt_threshold).astype(int)
cm = confusion_matrix(y_true_death, y_preds_opt)

plt.figure(figsize = (7, 6))
sns.heatmap(
    cm, 
    annot = True, 
    fmt = 'd', 
    cmap = 'YlOrRd', 
    xticklabels = ['Predicted : Safe', 'Predicted : High-Risk'], 
    yticklabels = ['Actual : Survive', 'Actual : Death']
)
plt.title(f'Final Clinical Risk Matrix\n(Optimized Threshold : {opt_threshold : .6f})')
plt.xlabel('Model Prediction')
plt.ylabel('Clinical Truth')
plt.show()

# 임상 지표 최종 리포트
print(f"📊 [Final Diagnostics : High-Risk Detection]")
print("=" * 50)
print(f"✅ 최적 위험 판단 임계값 : {opt_threshold : .6f}")
print(f"✅ 사망자 포착률 (Recall) : {recall[best_idx] : .4f}")
print(f"✅ 위험군 예측 정확도 (Precision) : {precision[best_idx] : .4f}")
print(f"✅ 유효 데이터 수 : {len(y_probs_arr)} / {len(y_probs_arr)}")
print("=" * 50)
print(classification_report(y_true_death, y_preds_opt, target_names = ['Normal', 'High-Risk']))

📊 [Final Diagnostics : High-Risk Detection]
✅ 최적 위험 판단 임계값 :  0.571289
✅ 사망자 포착률 (Recall) :  0.2727
✅ 위험군 예측 정확도 (Precision) :  0.1034
✅ 유효 데이터 수 : 1277 / 1277
              precision    recall  f1-score   support

      Normal       0.99      0.98      0.99      1266
   High-Risk       0.10      0.27      0.15        11

    accuracy                           0.97      1277
   macro avg       0.55      0.63      0.57      1277
weighted avg       0.99      0.97      0.98      1277



# 14. Precision-Recall Curve Analysis (정밀도-재현율 곡선 분석)

In [15]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

# AUPRC 점수 계산
auprc_score = average_precision_score(y_true_death, y_probs_death)

plt.figure(figsize = (8, 6))
plt.plot(recall, precision, color = 'red', lw = 2, label = f'P-R Curve (AUPRC = {auprc_score : .4f})')

# 현재 선택된 최적 지점 표시
plt.scatter(recall[best_idx], precision[best_idx], s = 100, color = 'black', 
            label = f'Selected Point (Recall : {recall[best_idx] : .2f})', zorder = 5)

plt.xlabel('Recall (사망자 포착률)')
plt.ylabel('Precision (위험군 예측 정확도)')
plt.title('Precision-Recall Curve for High-Risk Detection')
plt.legend(loc = 'upper right')
plt.grid(True, linestyle = '--', alpha = 0.5)
plt.show()

# 임상적 판단 보조 지표 출력
print("-" * 50)
print(f"임상적 분석 결과 요약")
print("-" * 50)
print(f" - 전체 사망 사례 중 탐지 성공 : {int(11 * recall[best_idx])} / 11 건")
print(f" - 경보 시 실제 위험군 확률  : {precision[best_idx] * 100 : .1f} %")
print(f" - 모델의 전반적 변별력(AUPRC) : {auprc_score : .4f}")
print("-" * 50)

--------------------------------------------------
임상적 분석 결과 요약
--------------------------------------------------
 - 전체 사망 사례 중 탐지 성공 : 3 / 11 건
 - 경보 시 실제 위험군 확률  :  10.3 %
 - 모델의 전반적 변별력(AUPRC) :  0.0358
--------------------------------------------------


# 15. Advanced Temporal Feature Engineering (시계열 추세 피처 추출)

In [16]:
import pandas as pd
import numpy as np

def extract_clinical_trends(dynamic_data) :
    """
    시계열 데이터에서 임상적 의미를 갖는 통계적 추세 추출
    dynamic_data shape : (samples, 3600, 8)
    """
    batch_size, seq_len, num_features = dynamic_data.shape
    
    # 1. 시계열 후반부(최근 상태)에 집중하기 위해 데이터를 분할
    # 전체 3600개 시퀀스 중 마지막 600개(약 10분)의 데이터 분석
    recent_data = dynamic_data[:, -600:, :]
    
    # 2. 기초 통계량 계산 (평균, 표준편차, 최소/최대)
    mean_vals = np.mean(recent_data, axis = 1)
    std_vals = np.std(recent_data, axis = 1)
    min_vals = np.min(recent_data, axis = 1)
    max_vals = np.max(recent_data, axis = 1)
    
    # 3. 혈역학적 변동성 (Coefficient of Variation)
    # 변동성이 크다는 것은 환자의 상태가 불안정함을 의미함
    cv_vals = std_vals / (mean_vals + 1e-8)
    
    # 4. 단순 기울기 계산 (최근 10분간의 변화량)
    # (종료 시점 값 - 시작 시점 값)
    slopes = recent_data[:, -1, :] - recent_data[:, 0, :]
    
    # 모든 특징 병합
    # 기존 6개의 정적 변수 뒤에 추가될 예정
    trend_features = np.hstack([
        mean_vals,   # 8개
        std_vals,    # 8개
        cv_vals,     # 8개
        slopes       # 8개
    ])
    
    return trend_features.astype(np.float32)

# 정적 데이터(Static)와 추세 데이터(Trend) 결합
X_trend = extract_clinical_trends(X_dynamic_scaled)
X_static_combined = np.hstack([X_static_scaled, X_trend])

print("-" * 50)
print(f"추세 피처 추출 완료")
print(f"기존 정적 피처 수 : {X_static_scaled.shape[1]}")
print(f"새로운 정적 피처 수 : {X_static_combined.shape[1]}")
print("-" * 50)

--------------------------------------------------
추세 피처 추출 완료
기존 정적 피처 수 : 6
새로운 정적 피처 수 : 38
--------------------------------------------------


# 11. Feature Importance Analysis (추세 피처 기여도 분석)

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score
import torch
from torch.amp import autocast

# 1. 배치 단위 추론 함수 (NaN 및 수치 안정성 보강)
def batch_predict_proba(stat_input, batch_size = 64) :
    model.eval()
    probs = []
    
    for i in range(0, len(stat_input), batch_size) :
        batch_stat = stat_input[i : i + batch_size]
        dummy_dyn = torch.zeros((len(batch_stat), 3600, 8)).to(device)
        stat_tensor = torch.FloatTensor(batch_stat).to(device)
        
        with torch.no_grad() :
            with autocast(device_type = 'cuda') :
                logits = model(dummy_dyn, stat_tensor)
                p = torch.sigmoid(torch.clamp(logits, min = -15.0, max = 15.0)).cpu().numpy()
            
            p = np.nan_to_num(p, nan = 0.0)
            probs.extend(p)
            
    return np.array(probs).flatten()

# 2. 검증 데이터 수집
all_stat, all_labels = [], []
for _, stat, labels in val_loader :
    all_stat.append(stat.numpy())
    all_labels.append(labels.numpy())

X_val_stat = np.vstack(all_stat)
y_val_death = 1 - np.vstack(all_labels).flatten()

# 3. Permutation Importance 계산
baseline_probs = batch_predict_proba(X_val_stat)
baseline_auc = roc_auc_score(y_val_death, baseline_probs)
importances = []

feature_names = ['Age', 'Sex', 'Height', 'Weight', 'BMI', 'ASA'] + \
                [f'Mean_{i}' for i in range(8)] + [f'Std_{i}' for i in range(8)] + \
                [f'CV_{i}' for i in range(8)] + [f'Slope_{i}' for i in range(8)]

for i in range(X_val_stat.shape[1]) :
    save_col = X_val_stat[:, i].copy()
    np.random.shuffle(X_val_stat[:, i])
    
    shuffled_probs = batch_predict_proba(X_val_stat)
    shuffled_auc = roc_auc_score(y_val_death, shuffled_probs)
    
    importances.append(baseline_auc - shuffled_auc)
    X_val_stat[:, i] = save_col
    
    if i % 10 == 0 :
        torch.cuda.empty_cache()

# 4. 결과 정리 및 텍스트 출력
importance_df = pd.DataFrame({'Feature' : feature_names, 'Importance' : importances})
importance_df = importance_df.sort_values(by = 'Importance', ascending = False)

print("\n" + "=" * 50)
print("✅ 분석 결과 리포트 : Feature Importance (Top 15)")
print("-" * 50)
print(importance_df.head(15).to_string(index = False))
print("=" * 50)

# 5. 시각화
plt.figure(figsize = (10, 8))
sns.barplot(data = importance_df.head(15), x = 'Importance', y = 'Feature', palette = 'magma')
plt.title('Clinical Feature Importance (AUROC Drop Analysis)')
plt.axvline(x = 0, color = 'black', lw = 1, linestyle = '--')
plt.grid(axis = 'x', alpha = 0.3)
plt.show()


✅ 분석 결과 리포트 : Feature Importance (Top 15)
--------------------------------------------------
Feature  Importance
Slope_7      0.0586
Slope_6      0.0499
  Std_4      0.0452
    Age      0.0358
    Sex      0.0322
   CV_6      0.0295
  Std_2      0.0273
    BMI      0.0208
   CV_0      0.0124
Slope_2      0.0112
  Std_0      0.0079
  Std_6      0.0013
 Mean_7      0.0006
   CV_4      0.0001
Slope_5      0.0000


# 12. Hybrid Ensemble Modeling (Transformer + XGBoost)

In [18]:
import xgboost as xgb
import numpy as np
from sklearn.metrics import average_precision_score

def extract_meta_features(loader) :
    model.eval()
    probs, stats, targets = [], [], []
    with torch.no_grad() :
        for dyn, stat, labels in loader :
            dyn, stat = dyn.to(device), stat.to(device)
            # Transformer 출력 추출 및 NaN 방어
            raw_output = model(dyn, stat)
            p = torch.sigmoid(torch.clamp(raw_output, min = -15.0, max = 15.0)).cpu().numpy()
            
            probs.extend(np.nan_to_num(p, nan = 0.5)) # NaN 발생 시 중립 확률(0.5) 부여
            stats.extend(stat.cpu().numpy())
            targets.extend(labels.cpu().numpy())
            
    return np.hstack([np.array(probs), np.array(stats)]), 1 - np.array(targets).flatten()

# 데이터 추출
X_train_ens, y_train_ens = extract_meta_features(train_loader)
X_val_ens, y_val_ens = extract_meta_features(val_loader)

# XGBoost 하이퍼파라미터 최적화
ens_model = xgb.XGBClassifier(
    n_estimators = 200,
    max_depth = 4,
    learning_rate = 0.03,
    scale_pos_weight = (len(y_train_ens) - sum(y_train_ens)) / sum(y_train_ens),
    random_state = 42,
    eval_metric = 'aucpr',
    tree_method = 'hist',
    device = 'cuda'
)
ens_model.fit(X_train_ens, y_train_ens)

y_ens_probs = ens_model.predict_proba(X_val_ens)[:, 1]
print(f"✅ 앙상블 모델 최종 AUPRC : {average_precision_score(y_val_ens, y_ens_probs) : .4f}")

✅ 앙상블 모델 최종 AUPRC :  0.0552


# 13. Data Integrity & Leakage Check

In [19]:
import numpy as np

# 1. 훈련 및 검증 데이터셋 인덱스 중복 여부 확인
overlap = set(train_idx).intersection(set(val_idx))
print(f"Dataset Index Overlap Check : {len(overlap)} 건")

# 2. 특징값(X)과 타겟(y) 간의 직접적 상관관계 점검
# 앙상블 모델에 입력된 Transformer 확률값과 실제 레이블 간의 관계 분석
correlation = np.corrcoef(X_train_ens[:, 0], y_train_ens)[0, 1]
print(f"Transformer Probability vs Target Correlation : {correlation : .4f}")

# 3. 데이터 분리 무결성 확인
print(f"Train Set Size : {len(train_idx)} 명")
print(f"Validation Set Size : {len(val_idx)} 명")
print(f"Total Unique Indices : {len(set(list(train_idx) + list(val_idx)))} / 6385")

Dataset Index Overlap Check : 0 건
Transformer Probability vs Target Correlation : -0.3490
Train Set Size : 5108 명
Validation Set Size : 1277 명
Total Unique Indices : 6385 / 6385


# 14. High-Sensitivity Threshold Optimization

In [20]:
import numpy as np
from sklearn.metrics import confusion_matrix

# Transformer 확률값 분포 확인 (correlation nan 원인 진단)
print(f"Transformer Prob Range : {X_train_ens[:, 0].min() : .6f} ~ {X_train_ens[:, 0].max() : .6f}")
print("-" * 65)

# 저확률 구간 정밀 탐색 (0.001 ~ 0.1)
fine_thresholds = np.linspace(0.001, 0.1, 10)

print(f"{'Threshold' : <12} | {'Recall (탐지 수)' : <20} | {'False Alarms (오경보)' : <20}")
print("-" * 65)

for th in fine_thresholds :
    y_ens_preds = (y_ens_probs >= th).astype(int)
    cm = confusion_matrix(y_val_ens, y_ens_preds)
    
    if cm.shape == (2, 2) :
        tn, fp, fn, tp = cm.ravel()
    else :
        tp = np.sum((y_val_ens == 1) & (y_ens_preds == 1))
        fp = np.sum((y_val_ens == 0) & (y_ens_preds == 1))
        fn = np.sum((y_val_ens == 1) & (y_ens_preds == 0))
        
    recall_pct = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0
    print(f"{th : <12.4f} | {int(tp) : >2} / 11 ({recall_pct : >5.1f}%) | {int(fp) : >15} 건")

Transformer Prob Range :  0.028047 ~  0.615782
-----------------------------------------------------------------
Threshold    | Recall (탐지 수)        | False Alarms (오경보)  
-----------------------------------------------------------------
0.0010       | 11 / 11 (100.0%) |            1266 건
0.0120       | 11 / 11 (100.0%) |            1195 건
0.0230       | 11 / 11 (100.0%) |            1104 건
0.0340       |  9 / 11 ( 81.8%) |             983 건
0.0450       |  9 / 11 ( 81.8%) |             902 건
0.0560       |  8 / 11 ( 72.7%) |             845 건
0.0670       |  7 / 11 ( 63.6%) |             798 건
0.0780       |  7 / 11 ( 63.6%) |             729 건
0.0890       |  7 / 11 ( 63.6%) |             650 건
0.1000       |  7 / 11 ( 63.6%) |             598 건


# 15. Search for Optimal Operating Point (0.1 - 0.5)

In [21]:
import numpy as np
from sklearn.metrics import confusion_matrix

# 0.1부터 0.5까지 조금 더 높은 임계값 구간 탐색
op_thresholds = np.linspace(0.1, 0.5, 9)

print(f"{'Threshold' : <12} | {'Recall (탐지 수)' : <20} | {'False Alarms (오경보)' : <20}")
print("-" * 65)

for th in op_thresholds :
    y_ens_preds = (y_ens_probs >= th).astype(int)
    cm = confusion_matrix(y_val_ens, y_ens_preds)
    
    if cm.shape == (2, 2) :
        tn, fp, fn, tp = cm.ravel()
    else :
        tp = np.sum((y_val_ens == 1) & (y_ens_preds == 1))
        fp = np.sum((y_val_ens == 0) & (y_ens_preds == 1))
        fn = np.sum((y_val_ens == 1) & (y_ens_preds == 0))
        
    recall_pct = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0
    print(f"{th : <12.3f} | {int(tp) : >2} / 11 ({recall_pct : >5.1f}%) | {int(fp) : >15} 건")

Threshold    | Recall (탐지 수)        | False Alarms (오경보)  
-----------------------------------------------------------------
0.100        |  7 / 11 ( 63.6%) |             598 건
0.150        |  5 / 11 ( 45.5%) |             434 건
0.200        |  3 / 11 ( 27.3%) |             340 건
0.250        |  1 / 11 (  9.1%) |             274 건
0.300        |  1 / 11 (  9.1%) |             216 건
0.350        |  1 / 11 (  9.1%) |             171 건
0.400        |  1 / 11 (  9.1%) |             138 건
0.450        |  1 / 11 (  9.1%) |              97 건
0.500        |  1 / 11 (  9.1%) |              69 건


# 16. Fine-tuning the 0.10 - 0.15 Operating Range

In [22]:
import numpy as np
from sklearn.metrics import confusion_matrix

# 0.1부터 0.15 사이를 0.01 단위로 정밀 분석
fine_op_thresholds = np.linspace(0.10, 0.15, 6)

print(f"{'Threshold' : <12} | {'Recall (탐지 수)' : <20} | {'False Alarms (오경보)' : <20}")
print("-" * 65)

for th in fine_op_thresholds :
    y_ens_preds = (y_ens_probs >= th).astype(int)
    cm = confusion_matrix(y_val_ens, y_ens_preds)
    
    if cm.shape == (2, 2) :
        tn, fp, fn, tp = cm.ravel()
    else :
        tp = np.sum((y_val_ens == 1) & (y_ens_preds == 1))
        fp = np.sum((y_val_ens == 0) & (y_ens_preds == 1))
        fn = np.sum((y_val_ens == 1) & (y_ens_preds == 0))
        
    recall_pct = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0
    print(f"{th : <12.3f} | {int(tp) : >2} / 11 ({recall_pct : >5.1f}%) | {int(fp) : >15} 건")

Threshold    | Recall (탐지 수)        | False Alarms (오경보)  
-----------------------------------------------------------------
0.100        |  7 / 11 ( 63.6%) |             598 건
0.110        |  7 / 11 ( 63.6%) |             550 건
0.120        |  7 / 11 ( 63.6%) |             505 건
0.130        |  7 / 11 ( 63.6%) |             471 건
0.140        |  6 / 11 ( 54.5%) |             454 건
0.150        |  5 / 11 ( 45.5%) |             434 건


# 17. Final Model Analysis & Feature Contribution

In [23]:
# XGBoost의 피처 중요도(Gain) 확인
importances = ens_model.get_booster().get_score(importance_type = 'gain')
sorted_importances = sorted(importances.items(), key = lambda x : x[1], reverse = True)

# 피처 이름 매핑 (f0: Transformer, f1-f6: Static, f7-f38: Trend)
feat_map = {'f0' : 'Transformer_Score'}
static_names = ['Age', 'Sex', 'Height', 'Weight', 'BMI', 'ASA']
for i, name in enumerate(static_names) : feat_map[f'f{i+1}'] = name
trend_names = [f'Mean_{i}' for i in range(8)] + [f'Std_{i}' for i in range(8)] + \
              [f'CV_{i}' for i in range(8)] + [f'Slope_{i}' for i in range(8)]
for i, name in enumerate(trend_names) : feat_map[f'f{i+7}'] = name

print("\n" + "=" * 60)
print("✅ 최종 앙상블 모델 핵심 피처 기여도 (Top 10)")
print("-" * 60)
for i in range(min(10, len(sorted_importances))) :
    feat_code, score = sorted_importances[i]
    feat_name = feat_map.get(feat_code, feat_code)
    print(f"Rank {i+1 : >2} : {feat_name : <20} | Score : {score : .4f}")
print("=" * 60)


✅ 최종 앙상블 모델 핵심 피처 기여도 (Top 10)
------------------------------------------------------------
Rank  1 : CV_4                 | Score :  211.7895
Rank  2 : Std_0                | Score :  87.1319
Rank  3 : Transformer_Score    | Score :  79.8201
Rank  4 : ASA                  | Score :  73.1972
Rank  5 : Slope_7              | Score :  68.5174
Rank  6 : CV_1                 | Score :  61.5936
Rank  7 : Mean_0               | Score :  60.1488
Rank  8 : Mean_7               | Score :  49.5356
Rank  9 : Sex                  | Score :  49.3166
Rank 10 : Height               | Score :  48.2489


# 18. Stratified 5-Fold Cross-Validation

In [24]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

# 1. 통합 데이터 준비
X_meta_all = np.vstack([X_train_ens, X_val_ens])
y_meta_all = np.concatenate([y_train_ens, y_val_ens])

skf = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
cv_auroc, cv_auprc = [], []

print("K-Fold Cross-Validation Progress")
print("-" * 50)

for fold, (t_idx, v_idx) in enumerate(skf.split(X_meta_all, y_meta_all)) :
    X_t, X_v = X_meta_all[t_idx], X_meta_all[v_idx]
    y_t, y_v = y_meta_all[t_idx], y_meta_all[v_idx]
    
    # 각 fold별 모델 학습
    fold_model = xgb.XGBClassifier(
        n_estimators = 200, max_depth = 4, learning_rate = 0.03,
        scale_pos_weight = (len(y_t) - sum(y_t)) / sum(y_t),
        random_state = 42, eval_metric = 'aucpr', tree_method = 'hist', device = 'cuda'
    )
    fold_model.fit(X_t, y_t)
    
    probs = fold_model.predict_proba(X_v)[:, 1]
    cv_auroc.append(roc_auc_score(y_v, probs))
    cv_auprc.append(average_precision_score(y_v, probs))
    
    print(f"Fold {fold+1} : AUROC = {cv_auroc[-1] : .4f} | AUPRC = {cv_auprc[-1] : .4f}")

print("-" * 50)
print(f"Final CV AUROC : {np.mean(cv_auroc) : .4f} (+/- {np.std(cv_auroc) : .4f})")
print(f"Final CV AUPRC : {np.mean(cv_auprc) : .4f} (+/- {np.std(cv_auprc) : .4f})")

K-Fold Cross-Validation Progress
--------------------------------------------------
Fold 1 : AUROC =  0.9934 | AUPRC =  0.9902
Fold 2 : AUROC =  0.9952 | AUPRC =  0.9918
Fold 3 : AUROC =  0.9880 | AUPRC =  0.9819
Fold 4 : AUROC =  0.9937 | AUPRC =  0.9884
Fold 5 : AUROC =  0.9906 | AUPRC =  0.9842
--------------------------------------------------
Final CV AUROC :  0.9922 (+/-  0.0026)
Final CV AUPRC :  0.9873 (+/-  0.0037)


# 19. Manual Probability Calibration

In [25]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibrationDisplay

# 1. 데이터 상태 진단
print(f"Data Diagnostic:")
print(f" - Prob Range: {y_ens_probs.min():.4f} ~ {y_ens_probs.max():.4f}")
print(f" - Unique Probs: {len(np.unique(y_ens_probs))}")
print(f" - Target Classes: {np.unique(y_val_ens)}")

# 2. Platt Scaling (Logistic Regression)
lr_calibrator = LogisticRegression(C=1e10)
lr_calibrator.fit(y_ens_probs.reshape(-1, 1), y_val_ens)
y_calibrated_probs = lr_calibrator.predict_proba(y_ens_probs.reshape(-1, 1))[:, 1]

# 3. 시각화 및 결과 확인
plt.figure(figsize=(7, 7))
ax = plt.gca()

# n_bins를 줄여서 출력 시도
CalibrationDisplay.from_predictions(y_val_ens, y_ens_probs, n_bins=5, name='Raw', ax=ax)
CalibrationDisplay.from_predictions(y_val_ens, y_calibrated_probs, n_bins=5, name='Calibrated', ax=ax)

ax.plot([0, 1], [0, 1], "k:", label="Perfect")
ax.set_title('Reliability Diagram')
ax.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 보정 후 상위 확률 확인
print(f"\nCalibrated Prob Range: {y_calibrated_probs.min():.4f} ~ {y_calibrated_probs.max():.4f}")

Data Diagnostic:
 - Prob Range: 0.0023 ~ 0.9812
 - Unique Probs: 1188
 - Target Classes: [0. 1.]

Calibrated Prob Range: 0.0070 ~ 0.0223


# 20. Clinical Error Analysis (TP vs FN)

In [26]:
# 1. 최종 예측 분류 (0.5가 아닌, 0.12 지점 등 실제 운영 임계값 기준)
threshold = 0.12 # 앞서 논의한 최적 지점
y_final_preds = (y_ens_probs >= threshold).astype(int)

# 2. 사망 환자 중 TP(탐지)와 FN(실패) 인덱스 추출
pos_indices = np.where(y_val_ens == 1)[0]
tp_indices = pos_indices[y_final_preds[pos_indices] == 1]
fn_indices = pos_indices[y_final_preds[pos_indices] == 0]

print(f"✅ [Detection Summary]")
print(f" - 전체 사망 환자 : {len(pos_indices)}명")
print(f" - 탐지 성공 (TP) : {len(tp_indices)}명")
print(f" - 탐지 실패 (FN) : {len(fn_indices)}명")
print("-" * 60)

# 3. 모델이 놓친 환자(FN)들의 특징값 분석
if len(fn_indices) > 0:
    # ASA(f7), CV_6(f30), CV_7(f31) 인덱스 기준
    target_feats = {'ASA': 7, 'CV_6': 30, 'CV_7': 31}
    print(f"{'Feature':<10} | {'TP Mean':<12} | {'FN Mean':<12}")
    print("-" * 40)
    for name, idx in target_feats.items():
        tp_m = X_val_ens[tp_indices, idx].mean()
        fn_m = X_val_ens[fn_indices, idx].mean()
        print(f"{name:<10} | {tp_m:<12.4f} | {fn_m:<12.4f}")
else:
    print("사망 환자를 전원 탐지하여 비교할 FN 케이스가 없습니다.")

✅ [Detection Summary]
 - 전체 사망 환자 : 11명
 - 탐지 성공 (TP) : 7명
 - 탐지 실패 (FN) : 4명
------------------------------------------------------------
Feature    | TP Mean      | FN Mean     
----------------------------------------
ASA        | -0.4546      | -0.2467     
CV_6       | 0.2694       | 0.5244      
CV_7       | -0.1525      | -0.2196     


# 21. Individual Explanation

In [36]:
import shap
import matplotlib.pyplot as plt
import numpy as np

# 1. 피처 이름(feat_names) 정의 (기존 유지)
static_feats = ['Age', 'Sex', 'Height', 'Weight', 'BMI', 'ASA']
dynamic_signals = ['MBP', 'SBP', 'DBP', 'HR', 'RR', 'SpO2', 'CO2', 'BT']
stats = ['Mean', 'Std', 'CV', 'Slope']
dynamic_feats = [f"{sig}_{stat}" for sig in dynamic_signals for stat in stats]
feat_names = ['Transformer_Score'] + static_feats + dynamic_feats

# 2. SHAP Explainer 초기화 및 값 계산 (기존 유지)
explainer = shap.TreeExplainer(ens_model)
shap_values = explainer.shap_values(X_val_ens)

# 3. 대상 환자(252번) 데이터 추출 (기존 유지)
target_idx = 252
if isinstance(shap_values, list):
    current_shap = shap_values[1][target_idx] if len(shap_values) > 1 else shap_values[0][target_idx]
else:
    current_shap = shap_values[target_idx]

# 4. 텍스트 리포트 출력 (기존 유지)
print(f"\n" + "="*55)
print(f"   [Individual Case Study] Patient Index: {target_idx}")
print("="*55)
top_idx = np.argsort(np.abs(current_shap))[::-1][:5]
print(f"{'Rank':<5} | {'Feature Name':<25} | {'SHAP Value':<10}")
print("-" * 55)
for i, idx in enumerate(top_idx):
    print(f"#{i+1:<4} | {feat_names[idx]:<25} | {current_shap[idx]:>10.4f}")
print("="*55 + "\n")

# 5. 시각화 개선 (수치 가려짐 방지 로직 적용)
indices = np.argsort(np.abs(current_shap))[-15:]
sorted_shaps = current_shap[indices]
sorted_names = [feat_names[i] for i in indices]

plt.close('all')
fig, ax = plt.subplots(figsize=(12, 9))
colors = ['#ff0051' if x > 0 else '#008bfb' for x in sorted_shaps]
bars = ax.barh(range(len(sorted_shaps)), sorted_shaps, color=colors, alpha=0.9)

ax.set_yticks(range(len(sorted_shaps)))
ax.set_yticklabels(sorted_names, fontsize=10)
ax.axvline(0, color='black', linewidth=1.2, zorder=3)

# 수치 라벨 가독성 개선 로직
max_val = np.max(np.abs(sorted_shaps))
offset = max_val * 0.02  # 막대 끝과의 간격 조절

for bar in bars:
    val = bar.get_width()
    # 양수/음수에 따라 텍스트 정렬(ha)과 위치(offset)를 다르게 설정
    if val >= 0:
        ha = 'left'
        x_pos = val + offset
    else:
        ha = 'right'
        x_pos = val - offset
    
    ax.text(x_pos, bar.get_y() + bar.get_height()/2, 
            f'{val:+.4f}', va='center', ha=ha,
            fontsize=10, fontweight='bold',
            color='#ff0051' if val > 0 else '#008bfb')

# 텍스트가 잘리지 않도록 x축 범위를 데이터보다 20% 더 넓게 설정
ax.set_xlim(-max_val * 1.2, max_val * 1.2)

plt.title(f"Clinical Evidence for Missed Case (Patient : {target_idx})", pad=25, fontsize=15)
plt.xlabel("SHAP Value (Impact on Mortality Prediction)", fontsize=11)
plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()

plt.savefig('missed_case_analysis.png', dpi=300)
plt.show()

print(f"✅ 분석 완료 : {target_idx}번 환자의 SHAP 분석 및 시각화가 완료되었습니다.")


   [Individual Case Study] Patient Index: 252
Rank  | Feature Name              | SHAP Value
-------------------------------------------------------
#1    | Transformer_Score         |    -1.1196
#2    | Age                       |    -0.8762
#3    | Height                    |    -0.8366
#4    | ASA                       |    -0.2594
#5    | RR_Std                    |    -0.1601

✅ 분석 완료 : 252번 환자의 SHAP 분석 및 시각화가 완료되었습니다.


# 22. Ablation Study : 정적 피처 제외 성능 비교

In [28]:
import xgboost as xgb
from sklearn.metrics import average_precision_score
import matplotlib.pyplot as plt

# 1. 피처 그룹 분리 (X_train_ens의 컬럼 구조 기준)
# f0: Transformer_Score, f1~f6: Static (Age, Sex, etc.), f7~f38: Vital Signs Trend
# 'Pure Dynamic' 그룹: 딥러닝 점수(f0) + 생체 신호 추세 피처(f7~f38)
dynamic_indices = [0] + list(range(7, X_train_ens.shape[1]))

X_train_dynamic = X_train_ens[:, dynamic_indices]
X_val_dynamic = X_val_ens[:, dynamic_indices]

# 2. 순수 시계열 기반 앙상블 모델 학습
# 기존 모델과 동일한 하이퍼파라미터 사용 (GPU 가속 활용)
dynamic_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.03,
    scale_pos_weight=(len(y_train_ens) - sum(y_train_ens)) / sum(y_train_ens),
    random_state=42, eval_metric='aucpr', tree_method='hist', device='cuda'
)
dynamic_model.fit(X_train_dynamic, y_train_ens)

# 3. 성능 비교 및 252번 환자 재검증
p_full = y_ens_probs # 기존 모델 확률
p_dyn = dynamic_model.predict_proba(X_val_dynamic)[:, 1] # 소거 모델 확률

auprc_full = average_precision_score(y_val_ens, p_full)
auprc_dyn = average_precision_score(y_val_ens, p_dyn)

print(f"--- Ablation Study Result ---")
print(f"Full Model AUPRC    : {auprc_full:.4f}")
print(f"Dynamic Only AUPRC : {auprc_dyn:.4f}")
print(f"AUPRC Drop          : {auprc_full - auprc_dyn:.4f}")
print("-" * 30)
print(f"Patient 252 Original Score : {p_full[252]:.4f}")
print(f"Patient 252 Dynamic Score : {p_dyn[252]:.4f}")
print(f"Result: {'Captured' if p_dyn[252] >= 0.120 else 'Still Missed'}")

--- Ablation Study Result ---
Full Model AUPRC    : 0.0552
Dynamic Only AUPRC : 0.0328
AUPRC Drop          : 0.0223
------------------------------
Patient 252 Original Score : 0.0328
Patient 252 Dynamic Score : 0.5090
Result: Captured


# 23. Decision Curve Analysis (DCA)

In [62]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick # X축 퍼센트 변환용
def calculate_net_benefit(y_true, y_prob, thresh):
    y_pred = (y_prob >= thresh).astype(int)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    n = len(y_true)
    if thresh <= 0 or thresh >= 1.0: return 0.0
    return (tp / n) - (fp / n) * (thresh / (1 - thresh))

# 분석 설정
thresholds = np.linspace(0.0001, 0.015, 200)
key_thresholds = [0.005, 0.008, 0.010] 
n = len(y_val_ens)
prevalence = np.mean(y_val_ens)

# 모델별 NB 계산
nb_full = np.array([calculate_net_benefit(y_val_ens, p_full, t) for t in thresholds])
nb_dyn = np.array([calculate_net_benefit(y_val_ens, p_dyn, t) for t in thresholds])
nb_all = np.array([(prevalence - (1 - prevalence) * (t / (1 - t))) for t in thresholds])
nb_none = np.zeros_like(thresholds)

# 수치 결과 출력
print(f"\n{'Threshold':<12} | {'Full Model':<15} | {'Dynamic Model':<15}")
print("-" * 50)
for kt in key_thresholds:
    val_full = calculate_net_benefit(y_val_ens, p_full, kt)
    val_dyn = calculate_net_benefit(y_val_ens, p_dyn, kt)
    print(f"{kt*100:>10.2f}%   | {val_full:>15.6f} | {val_dyn:>15.6f}")

# 시각화
fig, ax = plt.subplots(figsize=(10, 7))

ax.plot(thresholds, nb_full, color='#e41a1c', label='Full Model', linewidth=2)
ax.plot(thresholds, nb_dyn, color='#377eb8', label='Dynamic Model', linewidth=2, linestyle='--')
ax.plot(thresholds, nb_all, color='gray', label='Treat All', linewidth=1.5, alpha=0.5)
ax.plot(thresholds, nb_none, color='black', label='Treat None', linewidth=1.5)

# 주요 지점 수치 라벨링 (선과 겹치지 않게 지시선(Arrow) 사용)
for kt in key_thresholds:
    val_full = calculate_net_benefit(y_val_ens, p_full, kt)
    
    # 양수/음수에 따라 텍스트가 뻗어나가는 방향을 위/아래로 분리
    if val_full >= 0:
        xytext_offset = (15, 20)
    else:
        xytext_offset = (15, -20)
        
    ax.annotate(f'{val_full:.6f}', 
                xy=(kt, val_full), xycoords='data',
                xytext=xytext_offset, textcoords='offset points',
                color='#e41a1c', fontsize=9, fontweight='bold',
                arrowprops=dict(arrowstyle="-", color='gray', lw=0.6, alpha=0.7))

# 축 포맷팅 (논문 스타일)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0)) # 0.005 -> 0.5% 로 표시
ax.set_xlim(0.0, 0.015)

# 텅 빈 상단 여백 제거 (데이터 최고점에 맞춰 타이트하게 설정)
ax.set_ylim(-0.002, prevalence + 0.001)
ax.set_xlabel('Threshold Probability')
ax.set_ylabel('Net Benefit')
ax.set_title('Decision Curve Analysis')

# 디자인 정리 (테두리 제거 및 그리드 최소화)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper right', frameon=False)
ax.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig('dca_analysis.png', dpi=300)
plt.show()


Threshold    | Full Model      | Dynamic Model  
--------------------------------------------------
      0.50%   |        0.003679 |        0.003632
      0.80%   |        0.000764 |        0.000650
      1.00%   |       -0.001028 |       -0.001281


# 24. Robustness Test : 노이즈 주입을 통한 모델 강건성 검증

In [68]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score

# 재현성을 위한 시드 고정
np.random.seed(42)

def check_robustness(X, y, model, noise_level):
    # 생체 신호(VitalDB f7~f38) 영역에만 가우시안 노이즈 주입
    X_tmp = X.copy()
    sig_area = X_tmp[:, 7:]
    noise = np.random.normal(0, noise_level, size=sig_area.shape)
    X_tmp[:, 7:] += noise
    
    # 예측 및 AUPRC 계산
    y_prob = model.predict_proba(X_tmp)[:, 1]
    return average_precision_score(y, y_prob)

# 노이즈 강도 설정 (0% ~ 50%)
noise_levels = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]
results = []
print(f"{'Noise (%)':<10} | {'AUPRC'}")
print("-" * 25)

# 수준별 테스트 실행
for lvl in noise_levels:
    score = check_robustness(X_val_ens, y_val_ens, ens_model, lvl)
    results.append(score)
    print(f"{lvl*100:>8.1f}% | {score:.4f}")

# 시각화
plt.figure(figsize=(9, 6))
plt.plot(noise_levels, results, color='#ff0051', marker='o', lw=2, ms=7)
plt.fill_between(noise_levels, results, alpha=0.1, color='#ff0051')

# 각 포인트 위에 수치 라벨 추가
for i, score in enumerate(results):
    plt.text(noise_levels[i], score + (max(results) * 0.02), 
             f'{score:.4f}', 
             ha='center', va='bottom', 
             fontsize=9, fontweight='bold', color='#ff0051')
plt.title('Model Robustness: Noise Intensity vs AUPRC')
plt.xlabel('Noise Standard Deviation')
plt.ylabel('AUPRC Performance')

# 유병률이 낮은 데이터 특성상 미세한 변화가 잘 보이도록 Y축 조정
plt.ylim(min(results) * 0.8, max(results) * 1.2)
plt.grid(True, alpha=0.2, linestyle='--')
plt.tight_layout()
plt.savefig('robustness.png', dpi=300)
plt.show()

Noise (%)  | AUPRC
-------------------------
     0.0% | 0.0552
     5.0% | 0.0200
    10.0% | 0.0145
    20.0% | 0.0997
    30.0% | 0.0256
    50.0% | 0.0214


# 25. Inference Latency : 실시간 모니터링을 위한 추론 속도 측정

In [31]:
import time
import torch

# 1. 추론 속도 측정 함수 정의
def measure_latency(model, input_data, iterations=1000):
    # GPU 동기화 (정확한 시간 측정을 위함)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    start_time = time.time()
    
    with torch.no_grad():
        for _ in range(iterations):
            _ = model.predict_proba(input_data)
            
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        
    end_time = time.time()
    
    avg_latency = (end_time - start_time) / iterations
    return avg_latency

# 2. 단일 환자 데이터(1개 케이스) 추출
single_sample = X_val_ens[0:1]

# 3. 지연 시간 측정 실행
avg_latency = measure_latency(ens_model, single_sample)
fps = 1 / avg_latency # 초당 처리 가능한 샘플 수

print(f"\n" + "="*50)
print(f"  [System Performance: Inference Latency]")
print("="*50)
print(f" - Average Latency : {avg_latency*1000:.4f} ms")
print(f" - Throughput      : {fps:.2f} FPS (Inferences per second)")
print("-" * 50)
print(f" ✅ 수술실 실시간 모니터링 가능 여부 : {'적합(PASS)' if avg_latency < 0.1 else '검토 필요'}")
print("="*50)


  [System Performance: Inference Latency]
 - Average Latency : 1.3309 ms
 - Throughput      : 751.39 FPS (Inferences per second)
--------------------------------------------------
 ✅ 수술실 실시간 모니터링 가능 여부 : 적합(PASS)


# 26. Probability Calibration : 모델의 정직도 검증

In [32]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

# 1. 신뢰도 도표(Reliability Diagram) 데이터 생성
prob_true, prob_pred = calibration_curve(y_val_ens, y_ens_probs, n_bins=10)
brier_score = brier_score_loss(y_val_ens, y_ens_probs)

# 2. 시각화
plt.figure(figsize=(8, 8))
plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
plt.plot(prob_pred, prob_true, "s-", color='#ff0051', label=f"Ensemble Model (Brier={brier_score:.4f})")

plt.xlabel("Predicted Probability")
plt.ylabel("Actual Probability (Fraction of positives)")
plt.title("Probability Calibration Curve")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

print(f"✅ Brier Score: {brier_score:.4f} (0에 가까울수록 정밀함)")

✅ Brier Score: 0.0593 (0에 가까울수록 정밀함)


# 27. Global Interpretability : 전체 피처 영향력 분석

In [81]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm # 컬러맵 사용을 위한 임포트
import shap

# 1. SHAP 값 처리
if isinstance(shap_values, list):
    display_shap = shap_values[1]
else:
    display_shap = shap_values

# 2. 글로벌 중요도 계산 및 상위 15개 추출
global_importances = np.abs(display_shap).mean(0)
sorted_idx = np.argsort(global_importances)
top_15_idx = sorted_idx[-15:]

top_15_values = global_importances[top_15_idx]
top_15_names = np.array(feat_names)[top_15_idx]

# 3. 다채로운 막대 그래프 시각화
plt.close('all')
fig, ax = plt.subplots(figsize=(11, 8))

# 각 막대에 적용할 색상 생성 (plasma 컬러맵 사용)
colors = cm.plasma(np.linspace(0.2, 0.85, len(top_15_values)))

# 가로 막대 그래프 생성 (개별 색상 적용)
bars = ax.barh(range(len(top_15_values)), top_15_values, color=colors, alpha=0.85, height=0.7)

# Y축 레이블 및 막대 끝 수치 표기 (기존 유지)
ax.set_yticks(range(len(top_15_names)))
ax.set_yticklabels(top_15_names, fontsize=10)

max_val = top_15_values.max()
ax.set_xlim(0, max_val * 1.15)

for bar in bars:
    width = bar.get_width()
    ax.text(width + (max_val * 0.01), 
            bar.get_y() + bar.get_height()/2, 
            f'{width:.4f}', 
            va='center', ha='left', 
            fontsize=9, fontweight='bold', color='#333333')

# 디자인 정리 (깔끔한 스타일 유지)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', linestyle='--', alpha=0.3)

ax.set_xlabel("Mean |SHAP Value| (Average Impact on Prediction)", fontsize=11)
ax.set_title("Top 15 Global Feature Importance", fontsize=13, pad=20)

# 4. 저장 및 출력
plt.tight_layout()
plt.savefig('global_feature_importance_colorful.png', dpi=300, bbox_inches='tight')
plt.show()

# 5. 콘솔 수치 출력
print("\n" + "="*55)
print(f"✅ 분석 완료 : 상위 15개 주요 피처 중요도 (개별 색상 적용)")
print(f"   (시각화 파일 저장 : global_feature_importance_colorful.png)")
print("="*55)
print(f"[{'Feature Name':<30} | {'Mean |SHAP|':<12}]")
print("-" * 55)
for name, val in zip(reversed(top_15_names), reversed(top_15_values)):
    print(f"{name:<30} | {val:.6f}")
print("="*55 + "\n")


✅ 분석 완료 : 상위 15개 주요 피처 중요도 (개별 색상 적용)
   (시각화 파일 저장 : global_feature_importance_colorful.png)
[Feature Name                   | Mean |SHAP| ]
-------------------------------------------------------
Transformer_Score              | 1.026431
ASA                            | 0.463462
Height                         | 0.352988
Age                            | 0.314853
MBP_Mean                       | 0.268650
BMI                            | 0.241250
Weight                         | 0.096554
BT_CV                          | 0.077920
DBP_Slope                      | 0.072197
RR_Std                         | 0.064714
DBP_Mean                       | 0.039913
SBP_Slope                      | 0.039350
MBP_Std                        | 0.036737
HR_CV                          | 0.036518
Sex                            | 0.035874



# 28. Data Drift Simulation : 측정 장비 편향(Bias) 스트레스 테스트

In [69]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score

# 1. 측정 편향 시뮬레이션 및 평가
def evaluate_drift_bias(X, y, model, bias_ratio):
    X_drift = X.copy()
    # 생체 신호(VitalDB f7~f38) 영역에만 편향 주입
    X_drift[:, 7:] *= bias_ratio 
    probs = model.predict_proba(X_drift)[:, 1]
    return average_precision_score(y, probs)

# 2. 시나리오 설정 (수치 정확도 100% ~ 80%)
drift_scenarios = [1.0, 0.98, 0.95, 0.90, 0.85, 0.80]
drift_scores = []

print(f"\n{'Accuracy':<15} | {'AUPRC Score':<15} | {'Drop'}")
print("-" * 45)

# 3. 테스트 실행 및 결과 출력
for scenario in drift_scenarios:
    score = evaluate_drift_bias(X_val_ens, y_val_ens, ens_model, scenario)
    drift_scores.append(score)
    drop = drift_scores[0] - score
    print(f"{int(scenario*100):>9}% Accuracy | {score:>15.4f} | {drop:.4f}")

# 4. 시각화 및 수치 라벨링
plt.figure(figsize=(10, 6))
plt.plot(drift_scenarios, drift_scores, marker='s', color='#008bfb', linewidth=2, markersize=8)

# 그래프 내 수치 표시
for i, score in enumerate(drift_scores):
    plt.text(drift_scenarios[i], score + (max(drift_scores) * 0.005), 
             f'{score:.4f}', 
             ha='center', va='bottom', 
             fontsize=9, fontweight='bold', color='#008bfb')

plt.gca().invert_xaxis() # 100% -> 80% 흐름
plt.title('Performance vs Sensor Measurement Drift')
plt.xlabel('Measurement Accuracy (1.0 = Original)')
plt.ylabel('AUPRC Score')

# 수치가 잘리지 않도록 Y축 범위 조정
plt.ylim(min(drift_scores) * 0.98, max(drift_scores) * 1.05)
plt.grid(True, alpha=0.2, linestyle='--')
plt.tight_layout()
plt.savefig('data_drift_test.png', dpi=300)
plt.show()


Accuracy        | AUPRC Score     | Drop
---------------------------------------------
      100% Accuracy |          0.0552 | 0.0000
       98% Accuracy |          0.0556 | -0.0004
       95% Accuracy |          0.0557 | -0.0005
       90% Accuracy |          0.0556 | -0.0004
       85% Accuracy |          0.0555 | -0.0003
       80% Accuracy |          0.0568 | -0.0016
